In [54]:
from pathlib import Path
import pandas as pd
import torch
import math
import time

In [55]:
!pwd

/Users/ronit/Desktop/QeFEM/main/maxcut_results


In [6]:
# ------------------------------------------------------------
# Required base paths / objects
# ------------------------------------------------------------

GSET_DIR = Path("../data-gset/")

assert "GSET_DIR" in globals(), (
    "GSET_DIR is missing. Define the Gset data directory before running this cell."
)

print("=" * 80)
print("QEFEM V2: UNIFIED GSET METADATA")
print("=" * 80)

print("GSET_DIR:")
print(GSET_DIR.resolve())

QEFEM V2: UNIFIED GSET METADATA
GSET_DIR:
/Users/ronit/Desktop/QeFEM/main/data-gset


In [58]:
GSET_PARTITIONS = {
    1: ["G1", "G2", "G3", "G4", "G5", "G6", "G7", "G8", "G9", "G10"],
    2: ["G11", "G12", "G13"],
    3: ["G14", "G15", "G16", "G17", "G18", "G19", "G20", "G21"],
    4: ["G22", "G23", "G24", "G25", "G26", "G27", "G28", "G29", "G30", "G31"],
    5: ["G32", "G33", "G34"],
    6: ["G35", "G36", "G37", "G38", "G39", "G40", "G41", "G42"],
    7: ["G43", "G44", "G45", "G46", "G47"],
    8: ["G48", "G49", "G50"],
    9: ["G51", "G52", "G53", "G54"],
    10: ["G55", "G56"],
    11: ["G57"],
    12: ["G58", "G59"],
    13: ["G60", "G61"],
    14: ["G62"],
    15: ["G63", "G64"],
    16: ["G65"],
    17: ["G66"],
    18: ["G67", "G72"],
    19: ["G70"],
    20: ["G77"],
    21: ["G81"],
}

GSET_GRAPH_TYPES_BY_PARTITION = {
    1: "random",
    2: "toroidal",
    3: "plain",
    4: "random",
    5: "toroidal",
    6: "plain",
    7: "random",
    8: "toroidal",
    9: "plain",
    10: "plain",
    11: "toroid",
    12: "random",
    13: "plain",
    14: "toroid",
    15: "random",
    16: "toroid",
    17: "toroid",
    18: "toroid",
    19: "plain",
    20: "toroid",
    21: "toroid",
}


# ------------------------------------------------------------
# Storage mode by partition
# ------------------------------------------------------------

GSET_STORAGE_MODE_BY_PARTITION = {}

for partition_id in GSET_PARTITIONS:
    if partition_id <= 9:
        GSET_STORAGE_MODE_BY_PARTITION[partition_id] = "dense"
    else:
        GSET_STORAGE_MODE_BY_PARTITION[partition_id] = "edge_list"



# partition = batch number (1-21)
# graph = gset #
# graph type = plain/ toroid/ random
# storage mode = dense/ edge list


In [59]:
# ------------------------------------------------------------
# Build graph-level target table
# ------------------------------------------------------------

target_rows = []

for partition_id, graphs in GSET_PARTITIONS.items():
    graph_type = GSET_GRAPH_TYPES_BY_PARTITION[partition_id]
    storage_mode = GSET_STORAGE_MODE_BY_PARTITION[partition_id]

    for graph in graphs:
        graph_path = GSET_DIR / graph

        target_rows.append(
            {
                "partition": partition_id,
                "graph": graph,
                "graph_type": graph_type,
                "storage_mode": storage_mode,
                "path": graph_path,
                "exists": graph_path.exists(),
            }
        )

targets_df = pd.DataFrame(target_rows)

# ------------------------------------------------------------
# Print partition-level overview
# ------------------------------------------------------------

print("\nPartition overview:")

for partition_id, graphs in GSET_PARTITIONS.items():
    graph_type = GSET_GRAPH_TYPES_BY_PARTITION[partition_id]
    storage_mode = GSET_STORAGE_MODE_BY_PARTITION[partition_id]

    print(
        f"Partition {partition_id:2d}: "
        f"{graphs[0]}-{graphs[-1]} | "
        f"{len(graphs):2d} instances | "
        f"type={graph_type:} | "
        #f"path ={graph_path} | "
        f"storage={storage_mode}"
    )


Partition overview:
Partition  1: G1-G10 | 10 instances | type=random | storage=dense
Partition  2: G11-G13 |  3 instances | type=toroidal | storage=dense
Partition  3: G14-G21 |  8 instances | type=plain | storage=dense
Partition  4: G22-G31 | 10 instances | type=random | storage=dense
Partition  5: G32-G34 |  3 instances | type=toroidal | storage=dense
Partition  6: G35-G42 |  8 instances | type=plain | storage=dense
Partition  7: G43-G47 |  5 instances | type=random | storage=dense
Partition  8: G48-G50 |  3 instances | type=toroidal | storage=dense
Partition  9: G51-G54 |  4 instances | type=plain | storage=dense
Partition 10: G55-G56 |  2 instances | type=plain | storage=edge_list
Partition 11: G57-G57 |  1 instances | type=toroid | storage=edge_list
Partition 12: G58-G59 |  2 instances | type=random | storage=edge_list
Partition 13: G60-G61 |  2 instances | type=plain | storage=edge_list
Partition 14: G62-G62 |  1 instances | type=toroid | storage=edge_list
Partition 15: G63-G64

In [60]:
# ------------------------------------------------------------
# Print graph-level table
# ------------------------------------------------------------

print("\nGraph-level target table:")
display(targets_df)


# ------------------------------------------------------------
# File checks
# ------------------------------------------------------------

missing_graph_files = targets_df[
    targets_df["exists"] == False
]["graph"].tolist()

print("\nMissing graph files:")
print(missing_graph_files)

assert len(missing_graph_files) == 0, (
    f"Missing graph files: {missing_graph_files}"
)


# ------------------------------------------------------------
# Basic consistency checks
# ------------------------------------------------------------

all_graphs = targets_df["graph"].tolist()
unique_graphs = sorted(set(all_graphs))

EXPECTED_NUM_GSET_GRAPHS = sum(
    len(graphs)
    for graphs in GSET_PARTITIONS.values()
)

assert len(all_graphs) == EXPECTED_NUM_GSET_GRAPHS, (
    f"Expected {EXPECTED_NUM_GSET_GRAPHS} graph entries, got {len(all_graphs)}."
)

assert len(unique_graphs) == EXPECTED_NUM_GSET_GRAPHS, (
    "Duplicate graph names detected in GSET_PARTITIONS."
)

assert set(GSET_PARTITIONS.keys()) == set(range(1, 22)), (
    "Partitions should be exactly 1 through 21."
)

assert set(GSET_GRAPH_TYPES_BY_PARTITION.keys()) == set(GSET_PARTITIONS.keys()), (
    "Graph-type partition keys do not match GSET_PARTITIONS."
)

assert set(GSET_STORAGE_MODE_BY_PARTITION.keys()) == set(GSET_PARTITIONS.keys()), (
    "Storage-mode partition keys do not match GSET_PARTITIONS."
)

assert (
    targets_df[targets_df["partition"] <= 9]["storage_mode"] == "dense"
).all(), (
    "Partitions 1-9 should use dense storage."
)

assert (
    targets_df[targets_df["partition"] >= 10]["storage_mode"] == "edge_list"
).all(), (
    "Partitions 10-21 should use edge-list storage."
)


Graph-level target table:


,partition,graph,graph_type,storage_mode,path,exists
0,1,G1,random,dense,../data-gset/G1,True
1,1,G2,random,dense,../data-gset/G2,True
2,1,G3,random,dense,../data-gset/G3,True
3,1,G4,random,dense,../data-gset/G4,True
4,1,G5,random,dense,../data-gset/G5,True
...,...,...,...,...,...,...
66,18,G67,toroid,edge_list,../data-gset/G67,True
67,18,G72,toroid,edge_list,../data-gset/G72,True
68,19,G70,plain,edge_list,../data-gset/G70,True
69,20,G77,toroid,edge_list,../data-gset/G77,True



Missing graph files:
[]


In [61]:
assert "GSET_PARTITIONS" in globals(), (
    "Run Cell 01 first: GSET_PARTITIONS is missing."
)

assert "GSET_GRAPH_TYPES_BY_PARTITION" in globals(), (
    "Run Cell 01 first: GSET_GRAPH_TYPES_BY_PARTITION is missing."
)

assert "GSET_STORAGE_MODE_BY_PARTITION" in globals(), (
    "Run Cell 01 first: GSET_STORAGE_MODE_BY_PARTITION is missing."
)

assert "targets_df" in globals(), (
    "Run Cell 01 first: targets_df is missing."
)

assert "GSET_DIR" in globals(), (
    "Run Cell 01 first: GSET_DIR is missing."
)


# ------------------------------------------------------------
# Run groups
# ------------------------------------------------------------
# G1-G54 are grouped in threes of partitions.
# G55-G81 remain separated partitionwise.

RUN_GROUPS = {
    "small_1_3": [1, 2, 3],
    "small_4_6": [4, 5, 6],
    "small_7_9": [7, 8, 9],
    "big_10_12": [10,11,12],
    "big_13_15": [13,14,15],
    "big_16_18": [16,17,18],
    "big_19_21": [19,20,21],
}


# ------------------------------------------------------------
# Build run-group graph table
# ------------------------------------------------------------

run_group_rows = []

for run_group, partitions in RUN_GROUPS.items():
    for partition_id in partitions:
        graphs = GSET_PARTITIONS[partition_id]
        graph_type = GSET_GRAPH_TYPES_BY_PARTITION[partition_id]
        storage_mode = GSET_STORAGE_MODE_BY_PARTITION[partition_id]

        for graph in graphs:
            graph_path = GSET_DIR / graph

            run_group_rows.append(
                {
                    "run_group": run_group,
                    "partition": partition_id,
                    "graph": graph,
                    "graph_type": graph_type,
                    "storage_mode": storage_mode,
                    "path": graph_path,
                    "exists": graph_path.exists(),
                }
            )

run_groups_df = pd.DataFrame(run_group_rows)

In [62]:
# ------------------------------------------------------------
# Build run-group summary table
# ------------------------------------------------------------

run_group_summary_rows = []

for run_group, partitions in RUN_GROUPS.items():
    group_graphs = []

    for partition_id in partitions:
        group_graphs.extend(GSET_PARTITIONS[partition_id])

    group_types = [
        GSET_GRAPH_TYPES_BY_PARTITION[partition_id]
        for partition_id in partitions
    ]

    group_storage_modes = [
        GSET_STORAGE_MODE_BY_PARTITION[partition_id]
        for partition_id in partitions
    ]

    run_group_summary_rows.append(
        {
            "run_group": run_group,
            "partitions": partitions,
            "graphs_start": group_graphs[0],
            "graphs_end": group_graphs[-1],
            "num_graphs": len(group_graphs),
            "graph_types": sorted(set(group_types)),
            "storage_modes": sorted(set(group_storage_modes)),
            "graphs": group_graphs,
        }
    )

run_group_summary_df = pd.DataFrame(run_group_summary_rows)

In [63]:
# ------------------------------------------------------------
# Print run-group overview
# ------------------------------------------------------------

print("=" * 80)
print("RUN GROUPS")
print("=" * 80)

for _, row in run_group_summary_df.iterrows():
    print(
        f"{row['run_group']:<14s} | "
        f"partitions={row['partitions']} | "
        f"{row['graphs_start']}-{row['graphs_end']} | "
        f"{row['num_graphs']:2d} graphs | "
        f"types={row['graph_types']} | "
        f"storage={row['storage_modes']}"
    )

RUN GROUPS
small_1_3      | partitions=[1, 2, 3] | G1-G21 | 21 graphs | types=['plain', 'random', 'toroidal'] | storage=['dense']
small_4_6      | partitions=[4, 5, 6] | G22-G42 | 21 graphs | types=['plain', 'random', 'toroidal'] | storage=['dense']
small_7_9      | partitions=[7, 8, 9] | G43-G54 | 12 graphs | types=['plain', 'random', 'toroidal'] | storage=['dense']
big_10_12      | partitions=[10, 11, 12] | G55-G59 |  5 graphs | types=['plain', 'random', 'toroid'] | storage=['edge_list']
big_13_15      | partitions=[13, 14, 15] | G60-G64 |  5 graphs | types=['plain', 'random', 'toroid'] | storage=['edge_list']
big_16_18      | partitions=[16, 17, 18] | G65-G72 |  4 graphs | types=['toroid'] | storage=['edge_list']
big_19_21      | partitions=[19, 20, 21] | G70-G81 |  3 graphs | types=['plain', 'toroid'] | storage=['edge_list']


In [64]:
# ------------------------------------------------------------
# File checks
# ------------------------------------------------------------

missing_group_files = run_groups_df[
    run_groups_df["exists"] == False
]["graph"].tolist()

print("\nMissing graph files:")
print(missing_group_files)

assert len(missing_group_files) == 0, (
    f"Missing graph files in run groups: {missing_group_files}"
)


# ------------------------------------------------------------
# Consistency checks
# ------------------------------------------------------------

assert len(run_groups_df) == len(targets_df), (
    "Run-group graph count does not match targets_df."
)

assert sorted(run_groups_df["graph"].tolist()) == sorted(targets_df["graph"].tolist()), (
    "Run-group graph list does not match target graph list."
)

all_group_partitions = []

for partitions in RUN_GROUPS.values():
    all_group_partitions.extend(partitions)

assert sorted(all_group_partitions) == list(range(1, 22)), (
    "Run groups should cover partitions 1 through 21 exactly once."
)

assert len(all_group_partitions) == len(set(all_group_partitions)), (
    "Some partition appears in more than one run group."
)

assert (
    run_groups_df[run_groups_df["partition"] <= 9]["storage_mode"] == "dense"
).all(), (
    "Partitions 1-9 should use dense storage."
)

assert (
    run_groups_df[run_groups_df["partition"] >= 10]["storage_mode"] == "edge_list"
).all(), (
    "Partitions 10-21 should use edge-list storage."
)

small_run_groups = run_group_summary_df[
    run_group_summary_df["run_group"].str.startswith("small")
]

assert len(small_run_groups) == 3, (
    "Expected exactly three small run groups."
)


Missing graph files:
[]


In [65]:
print("\nRun-group summary dataframe:")
display(run_group_summary_df)


Run-group summary dataframe:


,run_group,partitions,graphs_start,graphs_end,num_graphs,graph_types,storage_modes,graphs
0,small_1_3,"[1, 2, 3]",G1,G21,21,"[plain, random, toroidal]",[dense],"[G1, G2, G3, G4, G5, G6, G7, G8, G9, G10, G11,..."
1,small_4_6,"[4, 5, 6]",G22,G42,21,"[plain, random, toroidal]",[dense],"[G22, G23, G24, G25, G26, G27, G28, G29, G30, ..."
2,small_7_9,"[7, 8, 9]",G43,G54,12,"[plain, random, toroidal]",[dense],"[G43, G44, G45, G46, G47, G48, G49, G50, G51, ..."
3,big_10_12,"[10, 11, 12]",G55,G59,5,"[plain, random, toroid]",[edge_list],"[G55, G56, G57, G58, G59]"
4,big_13_15,"[13, 14, 15]",G60,G64,5,"[plain, random, toroid]",[edge_list],"[G60, G61, G62, G63, G64]"
5,big_16_18,"[16, 17, 18]",G65,G72,4,[toroid],[edge_list],"[G65, G66, G67, G72]"
6,big_19_21,"[19, 20, 21]",G70,G81,3,"[plain, toroid]",[edge_list],"[G70, G77, G81]"


In [66]:
print("\nRun-group graph dataframe:")
display(run_groups_df)


Run-group graph dataframe:


,run_group,partition,graph,graph_type,storage_mode,path,exists
0,small_1_3,1,G1,random,dense,../data-gset/G1,True
1,small_1_3,1,G2,random,dense,../data-gset/G2,True
2,small_1_3,1,G3,random,dense,../data-gset/G3,True
3,small_1_3,1,G4,random,dense,../data-gset/G4,True
4,small_1_3,1,G5,random,dense,../data-gset/G5,True
...,...,...,...,...,...,...,...
66,big_16_18,18,G67,toroid,edge_list,../data-gset/G67,True
67,big_16_18,18,G72,toroid,edge_list,../data-gset/G72,True
68,big_19_21,19,G70,plain,edge_list,../data-gset/G70,True
69,big_19_21,20,G77,toroid,edge_list,../data-gset/G77,True


In [68]:
assert "targets_df" in globals(), (
    "Run Cell 01 first: targets_df is missing."
)

assert "GSET_DIR" in globals(), (
    "Run Cell 01 first: GSET_DIR is missing."
)


# ------------------------------------------------------------
# Locate validation file
# ------------------------------------------------------------

GSET_BEST_KNOWN_PATH = Path("gset_validation.txt")

if not GSET_BEST_KNOWN_PATH.exists():
    GSET_BEST_KNOWN_PATH = Path("../gset_validation.txt")

if not GSET_BEST_KNOWN_PATH.exists():
    GSET_BEST_KNOWN_PATH = GSET_DIR / "gset_validation.txt"

if not GSET_BEST_KNOWN_PATH.exists():
    GSET_BEST_KNOWN_PATH = Path("../data-gset/gset_validation.txt")


print("=" * 80)
print("BEST-KNOWN GSET VALIDATION VALUES")
print("=" * 80)

print("Validation path:")
print(GSET_BEST_KNOWN_PATH)

print("\nFile exists?")
print(GSET_BEST_KNOWN_PATH.exists())

assert GSET_BEST_KNOWN_PATH.exists(), (
    "Could not find gset_validation.txt. "
    "Set GSET_BEST_KNOWN_PATH manually."
)


# ------------------------------------------------------------
# Read validation file
# ------------------------------------------------------------

best_known_rows = []

with GSET_BEST_KNOWN_PATH.open("r") as f:
    for line_no, line in enumerate(f, start=1):
        line = line.strip()

        if not line:
            continue

        if line.startswith("#"):
            continue

        parts = line.replace(",", " ").split()

        if len(parts) < 2:
            continue

        graph_raw = parts[0]
        cut_raw = parts[1]

        if graph_raw.lower() in ["graph", "instance", "name"]:
            continue

        if graph_raw.startswith("G"):
            graph = graph_raw
        else:
            graph = "G" + graph_raw

        best_known_cut = float(cut_raw)

        best_known_rows.append(
            {
                "graph": graph,
                "best_known_cut": best_known_cut,
            }
        )

best_known_df = pd.DataFrame(best_known_rows)


# ------------------------------------------------------------
# Attach validation values to target table
# ------------------------------------------------------------

targets_with_validation_df = targets_df.merge(
    best_known_df,
    on="graph",
    how="left",
)

BEST-KNOWN GSET VALIDATION VALUES
Validation path:
gset_validation.txt

File exists?
True


In [69]:
# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\nLoaded best-known validation table:")
display(best_known_df)


Loaded best-known validation table:


,graph,best_known_cut
0,G1,11624.0
1,G2,11620.0
2,G3,11622.0
3,G4,11646.0
4,G5,11631.0
...,...,...
66,G67,6950.0
67,G70,9591.0
68,G72,7008.0
69,G77,9940.0


In [71]:
print("\nTargets with validation:")
display(targets_with_validation_df.drop(columns=["path","exists"]))


Targets with validation:


,partition,graph,graph_type,storage_mode,best_known_cut
0,1,G1,random,dense,11624.0
1,1,G2,random,dense,11620.0
2,1,G3,random,dense,11622.0
3,1,G4,random,dense,11646.0
4,1,G5,random,dense,11631.0
...,...,...,...,...,...
66,18,G67,toroid,edge_list,6950.0
67,18,G72,toroid,edge_list,7008.0
68,19,G70,plain,edge_list,9591.0
69,20,G77,toroid,edge_list,9940.0


In [73]:
# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------

missing_best_known = targets_with_validation_df[
    targets_with_validation_df["best_known_cut"].isna()
]["graph"].tolist()

print("\nMissing best-known values for selected graphs:")
print(missing_best_known)

assert len(missing_best_known) == 0, (
    f"Missing best-known cut values for: {missing_best_known}"
)

assert len(targets_with_validation_df) == len(targets_df), (
    "Validation merge changed the number of target rows."
)

assert targets_with_validation_df["graph"].is_unique, (
    "Duplicate graph names after validation merge."
)

assert targets_with_validation_df["best_known_cut"].notna().all(), (
    "Some selected graphs still have missing best-known cuts."
)


Missing best-known values for selected graphs:
[]


In [75]:
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# Input files
# ------------------------------------------------------------

V1_SMALL_RESULTS_PATH = Path("qefem_anneal_G1_G54_final_results.csv")
V1_BIG_RESULTS_PATH = Path("qefem_g55-g81_summary.csv")

print("=" * 80)
print("BUILDING CLEAN V1 BENCHMARK TABLE")
print("=" * 80)

print("Small V1 file:")
print(V1_SMALL_RESULTS_PATH.resolve())
print("exists:", V1_SMALL_RESULTS_PATH.exists())

print("\nBig V1 file:")
print(V1_BIG_RESULTS_PATH.resolve())
print("exists:", V1_BIG_RESULTS_PATH.exists())


# ------------------------------------------------------------
# Load small V1 results: G1-G54
# ------------------------------------------------------------

v1_small_raw_df = pd.read_csv(V1_SMALL_RESULTS_PATH)

v1_small_benchmark_df = v1_small_raw_df[
    [
        "instance",
        "nodes",
        "edges",
        "n_steps",
        "replicas",
        "lr",
        "best_cut",
        "best_value",
        "gap",
        "percent_accuracy",
        "best_replica",
        "runtime_sec",
    ]
].copy()

v1_small_benchmark_df = v1_small_benchmark_df.rename(
    columns={
        "best_cut": "v1_best",
        "best_value": "historical_best",
        "gap": "v1_gap",
        "percent_accuracy": "v1_percent_accuracy",
        "best_replica": "v1_best_replica",
        "runtime_sec": "v1_runtime_sec",
    }
)


# ------------------------------------------------------------
# Load big V1 results: G55-G81 selected subset
# ------------------------------------------------------------

v1_big_raw_df = pd.read_csv(V1_BIG_RESULTS_PATH)

v1_big_graph_df = v1_big_raw_df[
    v1_big_raw_df["graph"].notna()
].copy()

v1_big_benchmark_df = pd.DataFrame(
    {
        "instance": v1_big_graph_df["graph"],
        "nodes": v1_big_graph_df["N"].astype(int),
        "edges": v1_big_graph_df["M"].astype(int),
        "n_steps": v1_big_graph_df["steps"].astype(int),
        "replicas": v1_big_graph_df["replicas"].astype(int),
        "lr": v1_big_graph_df["lr"],
        "v1_best": v1_big_graph_df["qefem_best_cut"],
        "historical_best": v1_big_graph_df["best_known_cut"],
        "v1_gap": v1_big_graph_df["gap"],
        "v1_percent_accuracy": v1_big_graph_df["accuracy_percent"],
        "v1_best_replica": v1_big_graph_df["best_replica"].astype(int),
        "v1_runtime_sec": v1_big_graph_df["runtime_sec"],
    }
)


# ------------------------------------------------------------
# Combine small and big V1 benchmark tables
# ------------------------------------------------------------

benchmark_df = pd.concat(
    [
        v1_small_benchmark_df,
        v1_big_benchmark_df,
    ],
    ignore_index=True,
)

benchmark_df["graph_number"] = (
    benchmark_df["instance"]
    .str.replace("G", "", regex=False)
    .astype(int)
)

benchmark_df = benchmark_df.sort_values(
    "graph_number"
).reset_index(drop=True)


# ------------------------------------------------------------
# Add graph type and storage mode from targets_df
# ------------------------------------------------------------

benchmark_df = benchmark_df.merge(
    targets_df[
        [
            "graph",
            "partition",
            "graph_type",
            "storage_mode",
        ]
    ].rename(
        columns={
            "graph": "instance",
        }
    ),
    on="instance",
    how="left",
)

benchmark_df = benchmark_df[
    [
        "partition",
        "instance",
        "graph_type",
        "storage_mode",
        "nodes",
        "edges",
        "n_steps",
        "replicas",
        "lr",
        "v1_best",
        "historical_best",
        "v1_gap",
        "v1_percent_accuracy",
        "v1_best_replica",
        "v1_runtime_sec",
    ]
].copy()


# ------------------------------------------------------------
# Recompute simple consistency columns internally, but do not keep them
# ------------------------------------------------------------

gap_check = benchmark_df["historical_best"] - benchmark_df["v1_best"]
accuracy_check = 100.0 * benchmark_df["v1_best"] / benchmark_df["historical_best"]

max_gap_error = (gap_check - benchmark_df["v1_gap"]).abs().max()
max_accuracy_error = (accuracy_check - benchmark_df["v1_percent_accuracy"]).abs().max()

print("\nConsistency:")
print("max gap error:", float(max_gap_error))
print("max accuracy error:", float(max_accuracy_error))

BUILDING CLEAN V1 BENCHMARK TABLE
Small V1 file:
/Users/ronit/Desktop/QeFEM/main/maxcut_results/qefem_anneal_G1_G54_final_results.csv
exists: True

Big V1 file:
/Users/ronit/Desktop/QeFEM/main/maxcut_results/qefem_g55-g81_summary.csv
exists: True

Consistency:
max gap error: 0.0
max accuracy error: 1.4210854715202004e-14


In [76]:
# ------------------------------------------------------------
# Display clean table
# ------------------------------------------------------------

print("\nClean benchmark table:")
display(benchmark_df)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

BENCHMARK_PATH = Path("qefem_v1_benchmark_table.csv")

benchmark_df.to_csv(
    BENCHMARK_PATH,
    index=False,
)

print("\nSaved benchmark table:")
print(BENCHMARK_PATH.resolve())


Clean benchmark table:


,partition,instance,graph_type,storage_mode,nodes,edges,n_steps,replicas,lr,v1_best,historical_best,v1_gap,v1_percent_accuracy,v1_best_replica,v1_runtime_sec
0,1,G1,random,dense,800,19176,1000,128,0.01,11611.0,11624.0,13.0,99.888162,43,19.635248
1,1,G2,random,dense,800,19176,1000,128,0.01,11610.0,11620.0,10.0,99.913941,61,19.694396
2,1,G3,random,dense,800,19176,1000,128,0.01,11610.0,11622.0,12.0,99.896748,51,19.681336
3,1,G4,random,dense,800,19176,1000,128,0.01,11637.0,11646.0,9.0,99.922720,71,19.562205
4,1,G5,random,dense,800,19176,1000,128,0.01,11622.0,11631.0,9.0,99.922621,53,19.961461
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,18,G67,toroid,edge_list,10000,20000,1000,128,0.01,6806.0,6950.0,144.0,97.928058,35,51.017605
67,19,G70,plain,edge_list,10000,9999,1000,128,0.01,9412.0,9591.0,179.0,98.133667,62,36.532485
68,18,G72,toroid,edge_list,10000,20000,1000,128,0.01,6860.0,7008.0,148.0,97.888128,17,56.816828
69,20,G77,toroid,edge_list,14000,28000,1000,128,0.01,9722.0,9940.0,218.0,97.806841,106,62.788516



Saved benchmark table:
/Users/ronit/Desktop/QeFEM/main/maxcut_results/qefem_v1_benchmark_table.csv


In [78]:
# ------------------------------------------------------------
# Minimal useful summary
# ------------------------------------------------------------

print("Benchmark summary:")
print("graphs:", len(benchmark_df))
print("mean V1 accuracy %:", float(benchmark_df["v1_percent_accuracy"].mean()))
print("min V1 accuracy % :", float(benchmark_df["v1_percent_accuracy"].min()))
print("max V1 accuracy % :", float(benchmark_df["v1_percent_accuracy"].max()))
print("V1 gap-zero count :", int((benchmark_df["v1_gap"] == 0).sum()))

Benchmark summary:
graphs: 71
mean V1 accuracy %: 98.75137552535392
min V1 accuracy % : 95.17671517671518
max V1 accuracy % : 100.0
V1 gap-zero count : 2


# My targets for V2 run

In [37]:
# ============================================================
# Cell 06: Optimization settings
# ============================================================

import math
import torch


# ------------------------------------------------------------
# Device and dtype
# ------------------------------------------------------------

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

DTYPE = torch.float32


# ------------------------------------------------------------
# Main optimization settings
# ------------------------------------------------------------

N_REPLICAS = 128
N_STEPS = 2000
LOG_EVERY = 50
SEED = 7

ADAM_LR = 0.01


# ------------------------------------------------------------
# Temperature / beta settings
# ------------------------------------------------------------

T_MAX = 1.16
T_MIN = 6e-5


# ------------------------------------------------------------
# Quantum mixer gamma settings
# ------------------------------------------------------------

GAMMA_MAX = 1.0
GAMMA_MIN = 0.0

USE_GAMMA_PULSE = True

GAMMA_PULSE_AMP = 0.05
GAMMA_PULSE_CENTER_FRAC = 0.80
GAMMA_PULSE_WIDTH_FRAC = 0.03


# ------------------------------------------------------------
# Bloch initialization settings
# ------------------------------------------------------------

R_INIT = 1e-2
RAW_R_NOISE = 1e-3

THETA_CENTER = math.pi / 2
THETA_NOISE = 1e-2


# ------------------------------------------------------------
# Optional local polish settings
# ------------------------------------------------------------

USE_POLISH = True
POLISH_TOP_K = 8
POLISH_MAX_SWEEPS = 20


# ------------------------------------------------------------
# Print settings
# ------------------------------------------------------------

print("=" * 80)
print("OPTIMIZATION SETTINGS")
print("=" * 80)

print("device:", device)
print("dtype :", DTYPE)

print("\nOptimizer:")
print("optimizer: Adam")
print("lr:", ADAM_LR)

print("\nRun size:")
print("replicas:", N_REPLICAS)
print("steps:", N_STEPS)
print("log every:", LOG_EVERY)
print("seed:", SEED)

print("\nTemperature:")
print("T_MAX:", T_MAX)
print("T_MIN:", T_MIN)

print("\nGamma:")
print("GAMMA_MAX:", GAMMA_MAX)
print("GAMMA_MIN:", GAMMA_MIN)
print("USE_GAMMA_PULSE:", USE_GAMMA_PULSE)
print("GAMMA_PULSE_AMP:", GAMMA_PULSE_AMP)
print("GAMMA_PULSE_CENTER_FRAC:", GAMMA_PULSE_CENTER_FRAC)
print("GAMMA_PULSE_WIDTH_FRAC:", GAMMA_PULSE_WIDTH_FRAC)

print("\nInitialization:")
print("R_INIT:", R_INIT)
print("RAW_R_NOISE:", RAW_R_NOISE)
print("THETA_CENTER:", THETA_CENTER)
print("THETA_NOISE:", THETA_NOISE)

print("\nPolish:")
print("USE_POLISH:", USE_POLISH)
print("POLISH_TOP_K:", POLISH_TOP_K)
print("POLISH_MAX_SWEEPS:", POLISH_MAX_SWEEPS)


# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------

assert N_REPLICAS > 0, (
    "N_REPLICAS must be positive."
)

assert N_STEPS > 0, (
    "N_STEPS must be positive."
)

assert LOG_EVERY > 0, (
    "LOG_EVERY must be positive."
)

assert ADAM_LR > 0, (
    "ADAM_LR must be positive."
)

assert T_MAX > T_MIN > 0, (
    "Require T_MAX > T_MIN > 0."
)

assert GAMMA_MAX >= GAMMA_MIN >= 0, (
    "Require GAMMA_MAX >= GAMMA_MIN >= 0."
)

assert 0 < R_INIT < 1, (
    "R_INIT must lie in (0, 1)."
)

assert RAW_R_NOISE >= 0, (
    "RAW_R_NOISE must be non-negative."
)

assert THETA_NOISE >= 0, (
    "THETA_NOISE must be non-negative."
)

assert POLISH_TOP_K > 0, (
    "POLISH_TOP_K must be positive."
)

assert POLISH_MAX_SWEEPS >= 0, (
    "POLISH_MAX_SWEEPS must be non-negative."
)

OPTIMIZATION SETTINGS
device: mps
dtype : torch.float32

Optimizer:
optimizer: Adam
lr: 0.01

Run size:
replicas: 128
steps: 2000
log every: 50
seed: 7

Temperature:
T_MAX: 1.16
T_MIN: 6e-05

Gamma:
GAMMA_MAX: 1.0
GAMMA_MIN: 0.0
USE_GAMMA_PULSE: True
GAMMA_PULSE_AMP: 0.05
GAMMA_PULSE_CENTER_FRAC: 0.8
GAMMA_PULSE_WIDTH_FRAC: 0.03

Initialization:
R_INIT: 0.01
RAW_R_NOISE: 0.001
THETA_CENTER: 1.5707963267948966
THETA_NOISE: 0.01

Polish:
USE_POLISH: True
POLISH_TOP_K: 8
POLISH_MAX_SWEEPS: 20


In [39]:
# ============================================================
# Cell 07: QEFEM core functions
# ============================================================

assert "device" in globals(), (
    "Run Cell 06 first: device is missing."
)

assert "DTYPE" in globals(), (
    "Run Cell 06 first: DTYPE is missing."
)


# ------------------------------------------------------------
# Bloch parametrization
# ------------------------------------------------------------

def bloch_from_raw(raw_R, theta):
    R = torch.sigmoid(raw_R)

    rx = R * torch.sin(theta)
    rz = R * torch.cos(theta)

    return R, rx, rz


# ------------------------------------------------------------
# Quantum single-site entropy
# ------------------------------------------------------------

def quantum_entropy_from_R(R, eps=1e-12):
    lambda_plus = 0.5 * (1.0 + R)
    lambda_minus = 0.5 * (1.0 - R)

    lambda_plus = torch.clamp(
        lambda_plus,
        min=eps,
        max=1.0,
    )

    lambda_minus = torch.clamp(
        lambda_minus,
        min=eps,
        max=1.0,
    )

    entropy_per_site = -(
        lambda_plus * torch.log(lambda_plus)
        +
        lambda_minus * torch.log(lambda_minus)
    )

    entropy = entropy_per_site.sum(dim=1)

    return entropy


# ------------------------------------------------------------
# Transverse mixer / driver energy
# ------------------------------------------------------------

def transverse_driver_energy(rx, gamma):
    gamma = torch.as_tensor(
        gamma,
        device=rx.device,
        dtype=rx.dtype,
    )

    driver_energy = -gamma * rx.sum(dim=1)

    return driver_energy


# ------------------------------------------------------------
# Dense symmetric-W MaxCut functions
# ------------------------------------------------------------

def dense_expected_cut(W, rz):
    rz = rz.to(
        device=W.device,
        dtype=W.dtype,
    )

    quad = torch.einsum(
        "ri,ij,rj->r",
        rz,
        W,
        rz,
    )

    expected_cut = 0.25 * (
        W.sum() - quad
    )

    return expected_cut


def dense_discrete_cut(W, spins):
    single_input = False

    if spins.dim() == 1:
        spins = spins.unsqueeze(0)
        single_input = True

    spins = spins.to(
        device=W.device,
        dtype=W.dtype,
    )

    quad = torch.einsum(
        "ri,ij,rj->r",
        spins,
        W,
        spins,
    )

    cut = 0.25 * (
        W.sum() - quad
    )

    if single_input:
        cut = cut.squeeze(0)

    return cut


def dense_qefem_free_energy(W, raw_R, theta, beta, gamma):
    beta = torch.as_tensor(
        beta,
        device=raw_R.device,
        dtype=raw_R.dtype,
    )

    R, rx, rz = bloch_from_raw(
        raw_R,
        theta,
    )

    expected_cut = dense_expected_cut(
        W=W,
        rz=rz,
    )

    entropy = quantum_entropy_from_R(R)

    driver_energy = transverse_driver_energy(
        rx=rx,
        gamma=gamma,
    )

    free_energy = (
        -expected_cut
        + driver_energy
        - entropy / beta
    )

    aux = {
        "R": R,
        "rx": rx,
        "rz": rz,
        "expected_cut": expected_cut,
        "entropy": entropy,
        "driver_energy": driver_energy,
        "free_energy": free_energy,
        "beta": beta,
        "gamma": gamma,
    }

    return free_energy, aux


# ------------------------------------------------------------
# Edge-list MaxCut functions
# ------------------------------------------------------------

def edge_expected_cut(edge_i, edge_j, edge_w, rz):
    rz = rz.to(
        device=edge_w.device,
        dtype=edge_w.dtype,
    )

    zi = rz[:, edge_i]
    zj = rz[:, edge_j]

    expected_cut = 0.5 * (
        edge_w * (1.0 - zi * zj)
    ).sum(dim=1)

    return expected_cut


def edge_discrete_cut(edge_i, edge_j, edge_w, spins):
    single_input = False

    if spins.dim() == 1:
        spins = spins.unsqueeze(0)
        single_input = True

    spins = spins.to(
        device=edge_w.device,
        dtype=edge_w.dtype,
    )

    si = spins[:, edge_i]
    sj = spins[:, edge_j]

    cut = 0.5 * (
        edge_w * (1.0 - si * sj)
    ).sum(dim=1)

    if single_input:
        cut = cut.squeeze(0)

    return cut


def edge_qefem_free_energy(edge_i, edge_j, edge_w, raw_R, theta, beta, gamma):
    beta = torch.as_tensor(
        beta,
        device=raw_R.device,
        dtype=raw_R.dtype,
    )

    R, rx, rz = bloch_from_raw(
        raw_R,
        theta,
    )

    expected_cut = edge_expected_cut(
        edge_i=edge_i,
        edge_j=edge_j,
        edge_w=edge_w,
        rz=rz,
    )

    entropy = quantum_entropy_from_R(R)

    driver_energy = transverse_driver_energy(
        rx=rx,
        gamma=gamma,
    )

    free_energy = (
        -expected_cut
        + driver_energy
        - entropy / beta
    )

    aux = {
        "R": R,
        "rx": rx,
        "rz": rz,
        "expected_cut": expected_cut,
        "entropy": entropy,
        "driver_energy": driver_energy,
        "free_energy": free_energy,
        "beta": beta,
        "gamma": gamma,
    }

    return free_energy, aux


# ------------------------------------------------------------
# QEFEM variable initialization
# ------------------------------------------------------------

def initialize_qefem_variables(
    N,
    replicas,
    R_init,
    raw_R_noise,
    theta_center,
    theta_noise,
    seed,
    device=device,
    dtype=DTYPE,
):
    torch.manual_seed(seed)

    R_init_tensor = torch.tensor(
        R_init,
        device=device,
        dtype=dtype,
    )

    raw_R_init_value = torch.log(
        R_init_tensor / (1.0 - R_init_tensor)
    )

    raw_R = raw_R_init_value + raw_R_noise * torch.randn(
        replicas,
        N,
        device=device,
        dtype=dtype,
    )

    theta = theta_center + theta_noise * torch.randn(
        replicas,
        N,
        device=device,
        dtype=dtype,
    )

    raw_R.requires_grad_(True)
    theta.requires_grad_(True)

    return raw_R, theta


# ------------------------------------------------------------
# Hard readout
# ------------------------------------------------------------

def hard_readout_from_rz(rz):
    spins = torch.where(
        rz >= 0,
        torch.ones_like(rz),
        -torch.ones_like(rz),
    )

    return spins


# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------

test_raw_R = torch.zeros(
    2,
    5,
    device=device,
    dtype=DTYPE,
)

test_theta = torch.zeros(
    2,
    5,
    device=device,
    dtype=DTYPE,
)

test_R, test_rx, test_rz = bloch_from_raw(
    test_raw_R,
    test_theta,
)

test_entropy = quantum_entropy_from_R(test_R)
test_driver = transverse_driver_energy(test_rx, gamma=1.0)
test_spins = hard_readout_from_rz(test_rz)

assert test_R.shape == torch.Size([2, 5])
assert test_rx.shape == torch.Size([2, 5])
assert test_rz.shape == torch.Size([2, 5])

assert test_entropy.shape == torch.Size([2])
assert test_driver.shape == torch.Size([2])
assert test_spins.shape == torch.Size([2, 5])

assert torch.isfinite(test_R).all()
assert torch.isfinite(test_rx).all()
assert torch.isfinite(test_rz).all()
assert torch.isfinite(test_entropy).all()
assert torch.isfinite(test_driver).all()

assert set(torch.unique(test_spins).detach().cpu().tolist()).issubset({-1.0, 1.0})


print("=" * 80)
print("QEFEM CORE FUNCTIONS READY")
print("=" * 80)

print("Bloch:")
print("bloch_from_raw(raw_R, theta)")

print("\nEntropy / driver:")
print("quantum_entropy_from_R(R)")
print("transverse_driver_energy(rx, gamma)")

print("\nDense symmetric-W functions:")
print("dense_expected_cut(W, rz)")
print("dense_discrete_cut(W, spins)")
print("dense_qefem_free_energy(W, raw_R, theta, beta, gamma)")

print("\nEdge-list functions:")
print("edge_expected_cut(edge_i, edge_j, edge_w, rz)")
print("edge_discrete_cut(edge_i, edge_j, edge_w, spins)")
print("edge_qefem_free_energy(edge_i, edge_j, edge_w, raw_R, theta, beta, gamma)")

print("\nInitialization / readout:")
print("initialize_qefem_variables(...)")
print("hard_readout_from_rz(rz)")

QEFEM CORE FUNCTIONS READY
Bloch:
bloch_from_raw(raw_R, theta)

Entropy / driver:
quantum_entropy_from_R(R)
transverse_driver_energy(rx, gamma)

Dense symmetric-W functions:
dense_expected_cut(W, rz)
dense_discrete_cut(W, spins)
dense_qefem_free_energy(W, raw_R, theta, beta, gamma)

Edge-list functions:
edge_expected_cut(edge_i, edge_j, edge_w, rz)
edge_discrete_cut(edge_i, edge_j, edge_w, spins)
edge_qefem_free_energy(edge_i, edge_j, edge_w, raw_R, theta, beta, gamma)

Initialization / readout:
initialize_qefem_variables(...)
hard_readout_from_rz(rz)


In [41]:
# ============================================================
# Cell 08: Annealing schedules
# ============================================================

assert "device" in globals(), (
    "Run Cell 06 first: device is missing."
)

assert "DTYPE" in globals(), (
    "Run Cell 06 first: DTYPE is missing."
)

assert "N_STEPS" in globals(), (
    "Run Cell 06 first: N_STEPS is missing."
)

assert "T_MAX" in globals(), (
    "Run Cell 06 first: T_MAX is missing."
)

assert "T_MIN" in globals(), (
    "Run Cell 06 first: T_MIN is missing."
)

assert "GAMMA_MAX" in globals(), (
    "Run Cell 06 first: GAMMA_MAX is missing."
)

assert "GAMMA_MIN" in globals(), (
    "Run Cell 06 first: GAMMA_MIN is missing."
)

assert "USE_GAMMA_PULSE" in globals(), (
    "Run Cell 06 first: USE_GAMMA_PULSE is missing."
)

assert "GAMMA_PULSE_AMP" in globals(), (
    "Run Cell 06 first: GAMMA_PULSE_AMP is missing."
)

assert "GAMMA_PULSE_CENTER_FRAC" in globals(), (
    "Run Cell 06 first: GAMMA_PULSE_CENTER_FRAC is missing."
)

assert "GAMMA_PULSE_WIDTH_FRAC" in globals(), (
    "Run Cell 06 first: GAMMA_PULSE_WIDTH_FRAC is missing."
)


# ------------------------------------------------------------
# Temperature and beta schedules
# ------------------------------------------------------------

T_schedule = torch.linspace(
    T_MAX,
    T_MIN,
    N_STEPS,
    device=device,
    dtype=DTYPE,
)

beta_schedule = 1.0 / T_schedule


# ------------------------------------------------------------
# Base monotone gamma schedule
# ------------------------------------------------------------

gamma_base_schedule = torch.linspace(
    GAMMA_MAX,
    GAMMA_MIN,
    N_STEPS,
    device=device,
    dtype=DTYPE,
)


# ------------------------------------------------------------
# Late quantum mixer pulse
# ------------------------------------------------------------

step_index = torch.arange(
    N_STEPS,
    device=device,
    dtype=DTYPE,
)

gamma_pulse_center = GAMMA_PULSE_CENTER_FRAC * (N_STEPS - 1)
gamma_pulse_width = GAMMA_PULSE_WIDTH_FRAC * N_STEPS

gamma_pulse_schedule = GAMMA_PULSE_AMP * torch.exp(
    -0.5 * ((step_index - gamma_pulse_center) / gamma_pulse_width) ** 2
)

if USE_GAMMA_PULSE:
    gamma_schedule = gamma_base_schedule + gamma_pulse_schedule
else:
    gamma_schedule = gamma_base_schedule.clone()

gamma_schedule[-1] = GAMMA_MIN


# ------------------------------------------------------------
# Print schedule summary
# ------------------------------------------------------------

print("=" * 80)
print("ANNEALING SCHEDULES")
print("=" * 80)

print("N_STEPS:", N_STEPS)

print("\nTemperature schedule:")
print("T first :", float(T_schedule[0].detach().cpu()))
print("T middle:", float(T_schedule[N_STEPS // 2].detach().cpu()))
print("T last  :", float(T_schedule[-1].detach().cpu()))

print("\nBeta schedule:")
print("beta first :", float(beta_schedule[0].detach().cpu()))
print("beta middle:", float(beta_schedule[N_STEPS // 2].detach().cpu()))
print("beta last  :", float(beta_schedule[-1].detach().cpu()))

print("\nGamma base schedule:")
print("base gamma first :", float(gamma_base_schedule[0].detach().cpu()))
print("base gamma middle:", float(gamma_base_schedule[N_STEPS // 2].detach().cpu()))
print("base gamma last  :", float(gamma_base_schedule[-1].detach().cpu()))

print("\nGamma pulse schedule:")
print("use pulse:", USE_GAMMA_PULSE)
print("pulse amp:", GAMMA_PULSE_AMP)
print("pulse center step:", float(gamma_pulse_center))
print("pulse width steps:", float(gamma_pulse_width))
print("pulse max:", float(gamma_pulse_schedule.max().detach().cpu()))

print("\nFinal gamma schedule:")
print("gamma first :", float(gamma_schedule[0].detach().cpu()))
print("gamma middle:", float(gamma_schedule[N_STEPS // 2].detach().cpu()))
print("gamma max   :", float(gamma_schedule.max().detach().cpu()))
print("gamma last  :", float(gamma_schedule[-1].detach().cpu()))


# ------------------------------------------------------------
# Build schedule dataframe for inspection / saving later
# ------------------------------------------------------------

schedule_df = pd.DataFrame(
    {
        "step": torch.arange(1, N_STEPS + 1).detach().cpu().numpy(),
        "temperature": T_schedule.detach().cpu().numpy(),
        "beta": beta_schedule.detach().cpu().numpy(),
        "gamma_base": gamma_base_schedule.detach().cpu().numpy(),
        "gamma_pulse": gamma_pulse_schedule.detach().cpu().numpy(),
        "gamma": gamma_schedule.detach().cpu().numpy(),
    }
)

print("\nSchedule dataframe head:")
display(schedule_df.head())

print("\nSchedule dataframe tail:")
display(schedule_df.tail())


# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------

assert T_schedule.shape == torch.Size([N_STEPS])
assert beta_schedule.shape == torch.Size([N_STEPS])
assert gamma_base_schedule.shape == torch.Size([N_STEPS])
assert gamma_pulse_schedule.shape == torch.Size([N_STEPS])
assert gamma_schedule.shape == torch.Size([N_STEPS])

assert torch.all(T_schedule > 0), (
    "Temperature schedule must stay positive."
)

assert torch.all(beta_schedule > 0), (
    "Beta schedule must stay positive."
)

assert T_schedule[0] > T_schedule[-1], (
    "Temperature should decrease."
)

assert beta_schedule[0] < beta_schedule[-1], (
    "Beta should increase."
)

assert torch.all(gamma_schedule >= 0), (
    "Gamma schedule should be non-negative."
)

assert abs(float(gamma_schedule[-1].detach().cpu()) - GAMMA_MIN) < 1e-8, (
    "Final gamma should equal GAMMA_MIN."
)

assert len(schedule_df) == N_STEPS, (
    "schedule_df row count should match N_STEPS."
)

ANNEALING SCHEDULES
N_STEPS: 2000

Temperature schedule:
T first : 1.159999966621399
T middle: 0.5797398686408997
T last  : 5.999999848427251e-05

Beta schedule:
beta first : 0.8620690107345581
beta middle: 1.7249115705490112
beta last  : 16666.66796875

Gamma base schedule:
base gamma first : 1.0
base gamma middle: 0.4997498393058777
base gamma last  : 0.0

Gamma pulse schedule:
use pulse: True
pulse amp: 0.05
pulse center step: 1599.2
pulse width steps: 60.0
pulse max: 0.049999721348285675

Final gamma schedule:
gamma first : 1.0
gamma middle: 0.4997498393058777
gamma max   : 1.0
gamma last  : 0.0

Schedule dataframe head:


,step,temperature,beta,gamma_base,gamma_pulse,gamma
0,1,1.160000,0.862069,1.000000,0.0,1.000000
1,2,1.159420,0.862500,0.999500,0.0,0.999500
2,3,1.158839,0.862932,0.998999,0.0,0.998999
3,4,1.158259,0.863365,0.998499,0.0,0.998499
4,5,1.157679,0.863797,0.997999,0.0,0.997999



Schedule dataframe tail:


,step,temperature,beta,gamma_base,gamma_pulse,gamma
1995,1996,0.002381,419.997406,0.002001,1.776588e-11,0.002001
1996,1997,0.001801,555.316284,0.001501,1.591391e-11,0.001501
1997,1998,0.001220,819.360046,0.001000,1.425110e-11,0.001000
1998,1999,0.000640,1561.833496,0.000500,1.275844e-11,0.000500
1999,2000,0.000060,16666.667969,0.000000,1.141898e-11,0.000000


In [43]:
# ============================================================
# Cell 09: Graph tensor loaders
# ============================================================

assert "GSET_DIR" in globals(), (
    "Run Cell 01 first: GSET_DIR is missing."
)

assert "benchmark_df" in globals(), (
    "Run Cell 04 first: benchmark_df is missing."
)

assert "device" in globals(), (
    "Run Cell 06 first: device is missing."
)

assert "DTYPE" in globals(), (
    "Run Cell 06 first: DTYPE is missing."
)

assert "dense_discrete_cut" in globals(), (
    "Run Cell 07 first: dense_discrete_cut is missing."
)

assert "edge_discrete_cut" in globals(), (
    "Run Cell 07 first: edge_discrete_cut is missing."
)


# ------------------------------------------------------------
# Load Gset graph as edge-list tensors
# ------------------------------------------------------------

def load_gset_as_edge_list(graph, gset_dir=GSET_DIR, device=device, dtype=DTYPE):
    path = Path(gset_dir) / graph

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {graph} at:\n{path.resolve()}"
        )

    edge_i_list = []
    edge_j_list = []
    edge_w_list = []

    with path.open("r") as f:
        header = f.readline().strip().split()

        if len(header) != 2:
            raise ValueError(
                f"{graph}: bad header {header}. Expected: N M"
            )

        N_g = int(header[0])
        M_g = int(header[1])

        for line_no, line in enumerate(f, start=2):
            line = line.strip()

            if not line:
                continue

            parts = line.split()

            if len(parts) != 3:
                raise ValueError(
                    f"{graph}: bad edge line {line_no}: {line!r}"
                )

            i, j, w = map(int, parts)

            i -= 1
            j -= 1

            if not (0 <= i < N_g):
                raise IndexError(
                    f"{graph}: line {line_no}, i out of range after 0-indexing: {i}"
                )

            if not (0 <= j < N_g):
                raise IndexError(
                    f"{graph}: line {line_no}, j out of range after 0-indexing: {j}"
                )

            edge_i_list.append(i)
            edge_j_list.append(j)
            edge_w_list.append(float(w))

    if len(edge_i_list) != M_g:
        raise ValueError(
            f"{graph}: header says M={M_g}, but loaded {len(edge_i_list)} edges."
        )

    edge_i = torch.tensor(
        edge_i_list,
        device=device,
        dtype=torch.long,
    )

    edge_j = torch.tensor(
        edge_j_list,
        device=device,
        dtype=torch.long,
    )

    edge_w = torch.tensor(
        edge_w_list,
        device=device,
        dtype=dtype,
    )

    info = {
        "graph": graph,
        "N": N_g,
        "M": M_g,
        "positive_edges": int((edge_w > 0).sum().detach().cpu()),
        "negative_edges": int((edge_w < 0).sum().detach().cpu()),
        "zero_edges": int((edge_w == 0).sum().detach().cpu()),
        "weight_sum": float(edge_w.sum().detach().cpu()),
        "storage_mode": "edge_list",
    }

    return N_g, edge_i, edge_j, edge_w, info


# ------------------------------------------------------------
# Load Gset graph as dense symmetric W
# ------------------------------------------------------------

def load_gset_as_dense_W(graph, gset_dir=GSET_DIR, device=device, dtype=DTYPE):
    N_g, edge_i, edge_j, edge_w, edge_info = load_gset_as_edge_list(
        graph=graph,
        gset_dir=gset_dir,
        device=device,
        dtype=dtype,
    )

    W = torch.zeros(
        N_g,
        N_g,
        device=device,
        dtype=dtype,
    )

    W[edge_i, edge_j] = edge_w
    W[edge_j, edge_i] = edge_w

    diagonal_abs_sum = float(
        torch.diagonal(W).abs().sum().detach().cpu()
    )

    symmetry_error = float(
        (W - W.T).abs().max().detach().cpu()
    )

    info = dict(edge_info)
    info["storage_mode"] = "dense"
    info["diagonal_abs_sum"] = diagonal_abs_sum
    info["symmetry_error"] = symmetry_error
    info["dense_W_shape"] = tuple(W.shape)

    return N_g, W, info


# ------------------------------------------------------------
# Storage-aware graph loader
# ------------------------------------------------------------

def load_graph_for_run(graph, storage_mode, gset_dir=GSET_DIR, device=device, dtype=DTYPE):
    if storage_mode == "dense":
        N_g, W, info = load_gset_as_dense_W(
            graph=graph,
            gset_dir=gset_dir,
            device=device,
            dtype=dtype,
        )

        graph_data = {
            "graph": graph,
            "storage_mode": storage_mode,
            "N": N_g,
            "W": W,
            "info": info,
        }

    elif storage_mode == "edge_list":
        N_g, edge_i, edge_j, edge_w, info = load_gset_as_edge_list(
            graph=graph,
            gset_dir=gset_dir,
            device=device,
            dtype=dtype,
        )

        graph_data = {
            "graph": graph,
            "storage_mode": storage_mode,
            "N": N_g,
            "edge_i": edge_i,
            "edge_j": edge_j,
            "edge_w": edge_w,
            "info": info,
        }

    else:
        raise ValueError(
            f"Unknown storage_mode={storage_mode!r}. Expected 'dense' or 'edge_list'."
        )

    return graph_data


# ------------------------------------------------------------
# Sanity check: dense and edge-list hard cuts agree on one small graph
# ------------------------------------------------------------

dense_test_graph = benchmark_df[
    benchmark_df["storage_mode"] == "dense"
]["graph"].iloc[0]

print("=" * 80)
print("GRAPH LOADER SANITY CHECK")
print("=" * 80)

print("Dense test graph:")
print(dense_test_graph)

N_dense_test, W_dense_test, dense_info_test = load_gset_as_dense_W(
    dense_test_graph
)

N_edge_test, edge_i_test, edge_j_test, edge_w_test, edge_info_test = load_gset_as_edge_list(
    dense_test_graph
)

assert N_dense_test == N_edge_test, (
    "Dense and edge-list loaders disagree on N."
)

torch.manual_seed(SEED)

test_spins = torch.where(
    torch.randn(
        4,
        N_dense_test,
        device=device,
        dtype=DTYPE,
    ) >= 0,
    torch.ones(
        4,
        N_dense_test,
        device=device,
        dtype=DTYPE,
    ),
    -torch.ones(
        4,
        N_dense_test,
        device=device,
        dtype=DTYPE,
    ),
)

dense_test_cut = dense_discrete_cut(
    W=W_dense_test,
    spins=test_spins,
)

edge_test_cut = edge_discrete_cut(
    edge_i=edge_i_test,
    edge_j=edge_j_test,
    edge_w=edge_w_test,
    spins=test_spins,
)

cut_difference = (
    dense_test_cut - edge_test_cut
).abs()

print("\nDense cut values:")
print(dense_test_cut.detach().cpu().numpy())

print("\nEdge-list cut values:")
print(edge_test_cut.detach().cpu().numpy())

print("\nMax absolute difference:")
print(float(cut_difference.max().detach().cpu()))

assert torch.allclose(
    dense_test_cut,
    edge_test_cut,
    atol=1e-4,
), (
    "Dense and edge-list hard-cut formulas disagree."
)

assert dense_info_test["diagonal_abs_sum"] == 0.0, (
    "Dense W diagonal should be zero."
)

assert dense_info_test["symmetry_error"] == 0.0, (
    "Dense W should be symmetric."
)


# ------------------------------------------------------------
# Sanity check: load one large edge-list graph
# ------------------------------------------------------------

edge_test_graph = benchmark_df[
    benchmark_df["storage_mode"] == "edge_list"
]["graph"].iloc[0]

print("\nEdge-list test graph:")
print(edge_test_graph)

edge_graph_data_test = load_graph_for_run(
    graph=edge_test_graph,
    storage_mode="edge_list",
)

print("\nEdge-list loaded shapes:")
print("N:", edge_graph_data_test["N"])
print("edge_i:", edge_graph_data_test["edge_i"].shape)
print("edge_j:", edge_graph_data_test["edge_j"].shape)
print("edge_w:", edge_graph_data_test["edge_w"].shape)

assert edge_graph_data_test["edge_i"].shape == edge_graph_data_test["edge_j"].shape
assert edge_graph_data_test["edge_i"].shape == edge_graph_data_test["edge_w"].shape

assert torch.isfinite(edge_graph_data_test["edge_w"]).all(), (
    "edge_w contains non-finite values."
)


print("\nAvailable loaders:")
print("load_gset_as_dense_W(graph)")
print("load_gset_as_edge_list(graph)")
print("load_graph_for_run(graph, storage_mode)")

GRAPH LOADER SANITY CHECK
Dense test graph:
G1

Dense cut values:
[9572. 9436. 9585. 9563.]

Edge-list cut values:
[9572. 9436. 9585. 9563.]

Max absolute difference:
0.0

Edge-list test graph:
G55

Edge-list loaded shapes:
N: 5000
edge_i: torch.Size([12498])
edge_j: torch.Size([12498])
edge_w: torch.Size([12498])

Available loaders:
load_gset_as_dense_W(graph)
load_gset_as_edge_list(graph)
load_graph_for_run(graph, storage_mode)


In [44]:
# ============================================================
# Cell 10: Greedy one-flip polish functions
# ============================================================

assert "dense_discrete_cut" in globals(), (
    "Run Cell 07 first: dense_discrete_cut is missing."
)

assert "edge_discrete_cut" in globals(), (
    "Run Cell 07 first: edge_discrete_cut is missing."
)

assert "load_gset_as_dense_W" in globals(), (
    "Run Cell 09 first: load_gset_as_dense_W is missing."
)

assert "load_gset_as_edge_list" in globals(), (
    "Run Cell 09 first: load_gset_as_edge_list is missing."
)

assert "benchmark_df" in globals(), (
    "Run Cell 04 first: benchmark_df is missing."
)

assert "device" in globals(), (
    "Run Cell 06 first: device is missing."
)

assert "DTYPE" in globals(), (
    "Run Cell 06 first: DTYPE is missing."
)

assert "SEED" in globals(), (
    "Run Cell 06 first: SEED is missing."
)


# ------------------------------------------------------------
# Dense greedy one-flip polish
# ------------------------------------------------------------

def dense_greedy_one_flip_polish(W, spins, max_sweeps=20, tol=1e-8):
    single_input = False

    if spins.dim() == 1:
        spins = spins.unsqueeze(0)
        single_input = True

    spins = spins.clone().to(
        device=W.device,
        dtype=W.dtype,
    )

    sweeps_done = 0

    if max_sweeps <= 0:
        polished_cut = dense_discrete_cut(
            W=W,
            spins=spins,
        )

        if single_input:
            spins = spins.squeeze(0)
            polished_cut = polished_cut.squeeze(0)

        return spins, polished_cut, sweeps_done

    for sweep in range(max_sweeps):
        field = torch.matmul(
            spins,
            W,
        )

        delta_cut = spins * field

        best_delta, best_node = torch.max(
            delta_cut,
            dim=1,
        )

        improve_mask = best_delta > tol

        if not improve_mask.any():
            break

        improving_replicas = torch.where(improve_mask)[0]

        for replica in improving_replicas:
            node = best_node[replica]
            spins[replica, node] = -spins[replica, node]

        sweeps_done += 1

    polished_cut = dense_discrete_cut(
        W=W,
        spins=spins,
    )

    if single_input:
        spins = spins.squeeze(0)
        polished_cut = polished_cut.squeeze(0)

    return spins, polished_cut, sweeps_done


# ------------------------------------------------------------
# Edge-list greedy one-flip polish
# ------------------------------------------------------------

def edge_greedy_one_flip_polish(edge_i, edge_j, edge_w, spins, max_sweeps=20, tol=1e-8):
    single_input = False

    if spins.dim() == 1:
        spins = spins.unsqueeze(0)
        single_input = True

    spins = spins.clone().to(
        device=edge_w.device,
        dtype=edge_w.dtype,
    )

    R, N = spins.shape

    sweeps_done = 0

    if max_sweeps <= 0:
        polished_cut = edge_discrete_cut(
            edge_i=edge_i,
            edge_j=edge_j,
            edge_w=edge_w,
            spins=spins,
        )

        if single_input:
            spins = spins.squeeze(0)
            polished_cut = polished_cut.squeeze(0)

        return spins, polished_cut, sweeps_done

    for sweep in range(max_sweeps):
        improved_any = False

        for replica in range(R):
            s = spins[replica]

            field = torch.zeros(
                N,
                device=edge_w.device,
                dtype=edge_w.dtype,
            )

            field.index_add_(
                0,
                edge_i,
                edge_w * s[edge_j],
            )

            field.index_add_(
                0,
                edge_j,
                edge_w * s[edge_i],
            )

            delta_cut = s * field

            best_delta, best_node = torch.max(
                delta_cut,
                dim=0,
            )

            if best_delta > tol:
                s[best_node] = -s[best_node]
                improved_any = True

        if not improved_any:
            break

        sweeps_done += 1

    polished_cut = edge_discrete_cut(
        edge_i=edge_i,
        edge_j=edge_j,
        edge_w=edge_w,
        spins=spins,
    )

    if single_input:
        spins = spins.squeeze(0)
        polished_cut = polished_cut.squeeze(0)

    return spins, polished_cut, sweeps_done


# ------------------------------------------------------------
# Storage-aware polish wrapper
# ------------------------------------------------------------

def polish_spins_for_graph(graph_data, spins, max_sweeps=20):
    storage_mode = graph_data["storage_mode"]

    if storage_mode == "dense":
        polished_spins, polished_cut, sweeps_done = dense_greedy_one_flip_polish(
            W=graph_data["W"],
            spins=spins,
            max_sweeps=max_sweeps,
        )

    elif storage_mode == "edge_list":
        polished_spins, polished_cut, sweeps_done = edge_greedy_one_flip_polish(
            edge_i=graph_data["edge_i"],
            edge_j=graph_data["edge_j"],
            edge_w=graph_data["edge_w"],
            spins=spins,
            max_sweeps=max_sweeps,
        )

    else:
        raise ValueError(
            f"Unknown storage_mode={storage_mode!r}."
        )

    return polished_spins, polished_cut, sweeps_done


# ------------------------------------------------------------
# Sanity check on one small graph
# ------------------------------------------------------------

polish_test_graph = benchmark_df[
    benchmark_df["storage_mode"] == "dense"
]["graph"].iloc[0]

print("=" * 80)
print("GREEDY ONE-FLIP POLISH SANITY CHECK")
print("=" * 80)

print("Test graph:")
print(polish_test_graph)


# Dense version test

N_polish_dense, W_polish_test, dense_polish_info = load_gset_as_dense_W(
    polish_test_graph
)

torch.manual_seed(SEED)

test_spins_dense = torch.where(
    torch.randn(
        4,
        N_polish_dense,
        device=device,
        dtype=DTYPE,
    ) >= 0,
    torch.ones(
        4,
        N_polish_dense,
        device=device,
        dtype=DTYPE,
    ),
    -torch.ones(
        4,
        N_polish_dense,
        device=device,
        dtype=DTYPE,
    ),
)

dense_cut_before = dense_discrete_cut(
    W=W_polish_test,
    spins=test_spins_dense,
)

dense_spins_after, dense_cut_after, dense_sweeps_done = dense_greedy_one_flip_polish(
    W=W_polish_test,
    spins=test_spins_dense,
    max_sweeps=3,
)

print("\nDense polish:")
print("cut before:", dense_cut_before.detach().cpu().numpy())
print("cut after :", dense_cut_after.detach().cpu().numpy())
print("sweeps done:", dense_sweeps_done)

assert torch.all(dense_cut_after >= dense_cut_before - 1e-6), (
    "Dense polish decreased a cut value."
)


# Edge-list version test on same graph

N_polish_edge, edge_i_polish, edge_j_polish, edge_w_polish, edge_polish_info = load_gset_as_edge_list(
    polish_test_graph
)

assert N_polish_edge == N_polish_dense, (
    "Dense and edge-list test graph N mismatch."
)

edge_cut_before = edge_discrete_cut(
    edge_i=edge_i_polish,
    edge_j=edge_j_polish,
    edge_w=edge_w_polish,
    spins=test_spins_dense,
)

edge_spins_after, edge_cut_after, edge_sweeps_done = edge_greedy_one_flip_polish(
    edge_i=edge_i_polish,
    edge_j=edge_j_polish,
    edge_w=edge_w_polish,
    spins=test_spins_dense,
    max_sweeps=3,
)

print("\nEdge-list polish:")
print("cut before:", edge_cut_before.detach().cpu().numpy())
print("cut after :", edge_cut_after.detach().cpu().numpy())
print("sweeps done:", edge_sweeps_done)

assert torch.all(edge_cut_after >= edge_cut_before - 1e-6), (
    "Edge-list polish decreased a cut value."
)

assert torch.allclose(
    dense_cut_before,
    edge_cut_before,
    atol=1e-4,
), (
    "Dense and edge-list cuts disagree before polish."
)


print("\nAvailable polish functions:")
print("dense_greedy_one_flip_polish(W, spins, max_sweeps)")
print("edge_greedy_one_flip_polish(edge_i, edge_j, edge_w, spins, max_sweeps)")
print("polish_spins_for_graph(graph_data, spins, max_sweeps)")

GREEDY ONE-FLIP POLISH SANITY CHECK
Test graph:
G1

Dense polish:
cut before: [9572. 9436. 9585. 9563.]
cut after : [9635. 9499. 9641. 9627.]
sweeps done: 3

Edge-list polish:
cut before: [9572. 9436. 9585. 9563.]
cut after : [9635. 9499. 9641. 9627.]
sweeps done: 3

Available polish functions:
dense_greedy_one_flip_polish(W, spins, max_sweeps)
edge_greedy_one_flip_polish(edge_i, edge_j, edge_w, spins, max_sweeps)
polish_spins_for_graph(graph_data, spins, max_sweeps)


In [46]:
# ============================================================
# Cell 11: Select run group
# ============================================================

assert "RUN_GROUPS" in globals(), (
    "Run Cell 02 first: RUN_GROUPS is missing."
)

assert "run_groups_df" in globals(), (
    "Run Cell 02 first: run_groups_df is missing."
)

assert "run_group_summary_df" in globals(), (
    "Run Cell 02 first: run_group_summary_df is missing."
)

assert "benchmark_df" in globals(), (
    "Run Cell 04 first: benchmark_df is missing."
)

assert "v1_competitor_df" in globals(), (
    "Run Cell 05 first: v1_competitor_df is missing."
)


# ------------------------------------------------------------
# Choose run group here
# ------------------------------------------------------------
# Small groups:
#     "small_1_3"
#     "small_4_6"
#     "small_7_9"
#
# Big groups:
#     "partition_10"
#     ...
#     "partition_21"

SELECTED_RUN_GROUP = "small_1_3"


# ------------------------------------------------------------
# Validate selected run group
# ------------------------------------------------------------

assert SELECTED_RUN_GROUP in RUN_GROUPS, (
    f"Unknown run group: {SELECTED_RUN_GROUP}"
)

selected_partitions = RUN_GROUPS[SELECTED_RUN_GROUP]

selected_run_group_df = run_groups_df[
    run_groups_df["run_group"] == SELECTED_RUN_GROUP
].copy()

selected_benchmark_df = benchmark_df[
    benchmark_df["partition"].isin(selected_partitions)
].copy()

selected_v1_df = v1_competitor_df[
    v1_competitor_df["instance"].isin(
        selected_benchmark_df["graph"].tolist()
    )
].copy()


# ------------------------------------------------------------
# Sort selected tables by graph number
# ------------------------------------------------------------

selected_run_group_df["graph_number"] = (
    selected_run_group_df["graph"]
    .str.replace("G", "", regex=False)
    .astype(int)
)

selected_benchmark_df["graph_number"] = (
    selected_benchmark_df["graph"]
    .str.replace("G", "", regex=False)
    .astype(int)
)

selected_v1_df["graph_number"] = (
    selected_v1_df["instance"]
    .str.replace("G", "", regex=False)
    .astype(int)
)

selected_run_group_df = selected_run_group_df.sort_values(
    "graph_number"
).reset_index(drop=True)

selected_benchmark_df = selected_benchmark_df.sort_values(
    "graph_number"
).reset_index(drop=True)

selected_v1_df = selected_v1_df.sort_values(
    "graph_number"
).reset_index(drop=True)


# ------------------------------------------------------------
# Print selected run group
# ------------------------------------------------------------

print("=" * 80)
print("SELECTED RUN GROUP")
print("=" * 80)

print("Run group:", SELECTED_RUN_GROUP)
print("Partitions:", selected_partitions)

print("\nGraphs:")
print(selected_run_group_df["graph"].tolist())

print("\nNumber of graphs:")
print(len(selected_run_group_df))

print("\nStorage modes:")
print(selected_run_group_df["storage_mode"].unique().tolist())

print("\nGraph types:")
print(selected_run_group_df["graph_type"].unique().tolist())


# ------------------------------------------------------------
# Display selected benchmark and V1 competitor tables
# ------------------------------------------------------------

print("\nSelected benchmark table:")
display(
    selected_benchmark_df[
        [
            "partition",
            "graph",
            "graph_type",
            "storage_mode",
            "N",
            "M",
            "density_upper_percent",
            "best_known_cut",
            "best_known_cut_per_edge",
        ]
    ]
)

print("\nSelected V1 competitor table:")
display(
    selected_v1_df[
        [
            "instance",
            "nodes",
            "edges",
            "n_steps",
            "replicas",
            "lr",
            "best_cut",
            "best_value",
            "gap",
            "percent_accuracy",
            "best_replica",
            "runtime_sec",
        ]
    ]
)


# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------

assert len(selected_run_group_df) > 0, (
    "Selected run group has no graphs."
)

assert len(selected_run_group_df) == len(selected_benchmark_df), (
    "Selected run-group table and benchmark table disagree in row count."
)

assert len(selected_v1_df) == len(selected_benchmark_df), (
    "Selected V1 table and benchmark table disagree in row count."
)

assert selected_run_group_df["graph"].tolist() == selected_benchmark_df["graph"].tolist(), (
    "Selected run-group graphs and benchmark graphs are not aligned."
)

assert selected_benchmark_df["graph"].tolist() == selected_v1_df["instance"].tolist(), (
    "Selected benchmark graphs and V1 competitor graphs are not aligned."
)

assert selected_benchmark_df["best_known_cut"].notna().all(), (
    "Some selected graphs are missing best-known cut values."
)

assert selected_v1_df["best_cut"].notna().all(), (
    "Some selected graphs are missing V1 best cuts."
)

assert selected_run_group_df["exists"].all(), (
    "Some selected graph files are missing."
)

SELECTED RUN GROUP
Run group: small_1_3
Partitions: [1, 2, 3]

Graphs:
['G1', 'G2', 'G3', 'G4', 'G5', 'G6', 'G7', 'G8', 'G9', 'G10', 'G11', 'G12', 'G13', 'G14', 'G15', 'G16', 'G17', 'G18', 'G19', 'G20', 'G21']

Number of graphs:
21

Storage modes:
['dense']

Graph types:
['random', 'toroidal', 'plain']

Selected benchmark table:


,partition,graph,graph_type,storage_mode,N,M,density_upper_percent,best_known_cut,best_known_cut_per_edge
0,1,G1,random,dense,800,19176,6.000000,11624.0,0.606174
1,1,G2,random,dense,800,19176,6.000000,11620.0,0.605966
2,1,G3,random,dense,800,19176,6.000000,11622.0,0.606070
3,1,G4,random,dense,800,19176,6.000000,11646.0,0.607322
4,1,G5,random,dense,800,19176,6.000000,11631.0,0.606539
5,1,G6,random,dense,800,19176,6.000000,2178.0,0.113579
6,1,G7,random,dense,800,19176,6.000000,2006.0,0.104610
7,1,G8,random,dense,800,19176,6.000000,2005.0,0.104558
8,1,G9,random,dense,800,19176,6.000000,2054.0,0.107113
9,1,G10,random,dense,800,19176,6.000000,2000.0,0.104297



Selected V1 competitor table:


,instance,nodes,edges,n_steps,replicas,lr,best_cut,best_value,gap,percent_accuracy,best_replica,runtime_sec
0,G1,800,19176,1000,128,0.01,11611.0,11624.0,13.0,99.888162,43,19.635248
1,G2,800,19176,1000,128,0.01,11610.0,11620.0,10.0,99.913941,61,19.694396
2,G3,800,19176,1000,128,0.01,11610.0,11622.0,12.0,99.896748,51,19.681336
3,G4,800,19176,1000,128,0.01,11637.0,11646.0,9.0,99.922720,71,19.562205
4,G5,800,19176,1000,128,0.01,11622.0,11631.0,9.0,99.922621,53,19.961461
5,G6,800,19176,1000,128,0.01,2173.0,2178.0,5.0,99.770432,59,19.691165
6,G7,800,19176,1000,128,0.01,1992.0,2006.0,14.0,99.302094,74,20.118622
7,G8,800,19176,1000,128,0.01,1991.0,2005.0,14.0,99.301746,102,19.074841
8,G9,800,19176,1000,128,0.01,2041.0,2054.0,13.0,99.367089,94,19.802351
9,G10,800,19176,1000,128,0.01,1992.0,2000.0,8.0,99.600000,122,19.633554


In [49]:
# ============================================================
# Cell 12: Single-graph QEFEM runner
# ============================================================

assert "load_graph_for_run" in globals(), (
    "Run Cell 09 first: load_graph_for_run is missing."
)

assert "dense_qefem_free_energy" in globals(), (
    "Run Cell 07 first: dense_qefem_free_energy is missing."
)

assert "edge_qefem_free_energy" in globals(), (
    "Run Cell 07 first: edge_qefem_free_energy is missing."
)

assert "dense_discrete_cut" in globals(), (
    "Run Cell 07 first: dense_discrete_cut is missing."
)

assert "edge_discrete_cut" in globals(), (
    "Run Cell 07 first: edge_discrete_cut is missing."
)

assert "initialize_qefem_variables" in globals(), (
    "Run Cell 07 first: initialize_qefem_variables is missing."
)

assert "hard_readout_from_rz" in globals(), (
    "Run Cell 07 first: hard_readout_from_rz is missing."
)

assert "polish_spins_for_graph" in globals(), (
    "Run Cell 10 first: polish_spins_for_graph is missing."
)

assert "beta_schedule" in globals(), (
    "Run Cell 08 first: beta_schedule is missing."
)

assert "gamma_schedule" in globals(), (
    "Run Cell 08 first: gamma_schedule is missing."
)

assert "benchmark_df" in globals(), (
    "Run Cell 04 first: benchmark_df is missing."
)

assert "v1_competitor_df" in globals(), (
    "Run Cell 05 first: v1_competitor_df is missing."
)


# ------------------------------------------------------------
# Utility: hard cut from graph data
# ------------------------------------------------------------

def hard_cut_for_graph_data(graph_data, spins):
    storage_mode = graph_data["storage_mode"]

    if storage_mode == "dense":
        cut = dense_discrete_cut(
            W=graph_data["W"],
            spins=spins,
        )

    elif storage_mode == "edge_list":
        cut = edge_discrete_cut(
            edge_i=graph_data["edge_i"],
            edge_j=graph_data["edge_j"],
            edge_w=graph_data["edge_w"],
            spins=spins,
        )

    else:
        raise ValueError(
            f"Unknown storage_mode={storage_mode!r}."
        )

    return cut


# ------------------------------------------------------------
# Utility: free energy from graph data
# ------------------------------------------------------------

def qefem_free_energy_for_graph_data(graph_data, raw_R, theta, beta, gamma):
    storage_mode = graph_data["storage_mode"]

    if storage_mode == "dense":
        F, aux = dense_qefem_free_energy(
            W=graph_data["W"],
            raw_R=raw_R,
            theta=theta,
            beta=beta,
            gamma=gamma,
        )

    elif storage_mode == "edge_list":
        F, aux = edge_qefem_free_energy(
            edge_i=graph_data["edge_i"],
            edge_j=graph_data["edge_j"],
            edge_w=graph_data["edge_w"],
            raw_R=raw_R,
            theta=theta,
            beta=beta,
            gamma=gamma,
        )

    else:
        raise ValueError(
            f"Unknown storage_mode={storage_mode!r}."
        )

    return F, aux


# ------------------------------------------------------------
# Main single-graph runner
# ------------------------------------------------------------

def run_qefem_on_single_graph(
    graph,
    storage_mode,
    best_known_cut,
    v1_best_cut,
    v1_gap,
    v1_percent_accuracy,
    seed_offset=0,
):
    print("=" * 80)
    print("RUNNING QEFEM ON SINGLE GRAPH")
    print("=" * 80)
    print("graph:", graph)
    print("storage_mode:", storage_mode)

    start_time = time.time()

    graph_data = load_graph_for_run(
        graph=graph,
        storage_mode=storage_mode,
        device=device,
        dtype=DTYPE,
    )

    N_g = graph_data["N"]

    print("N:", N_g)
    print("best_known_cut:", best_known_cut)
    print("v1_best_cut:", v1_best_cut)

    raw_R, theta = initialize_qefem_variables(
        N=N_g,
        replicas=N_REPLICAS,
        R_init=R_INIT,
        raw_R_noise=RAW_R_NOISE,
        theta_center=THETA_CENTER,
        theta_noise=THETA_NOISE,
        seed=SEED + seed_offset,
        device=device,
        dtype=DTYPE,
    )

    optimizer = torch.optim.Adam(
        [
            raw_R,
            theta,
        ],
        lr=ADAM_LR,
    )

    best_raw_cut_value = -float("inf")
    best_raw_step = None
    best_raw_replica = None
    best_raw_spins = None

    log_rows = []

    for step in range(N_STEPS):
        beta_t = beta_schedule[step]
        gamma_t = gamma_schedule[step]

        optimizer.zero_grad()

        F, aux = qefem_free_energy_for_graph_data(
            graph_data=graph_data,
            raw_R=raw_R,
            theta=theta,
            beta=beta_t,
            gamma=gamma_t,
        )

        loss = F.mean()

        loss.backward()
        optimizer.step()

        with torch.no_grad():
            R = aux["R"]
            rx = aux["rx"]
            rz = aux["rz"]

            spins = hard_readout_from_rz(rz)

            raw_cut = hard_cut_for_graph_data(
                graph_data=graph_data,
                spins=spins,
            )

            step_best_cut, step_best_replica = torch.max(
                raw_cut,
                dim=0,
            )

            step_best_cut_value = float(
                step_best_cut.detach().cpu()
            )

            if step_best_cut_value > best_raw_cut_value:
                best_raw_cut_value = step_best_cut_value
                best_raw_step = step + 1
                best_raw_replica = int(
                    step_best_replica.detach().cpu()
                )

                best_raw_spins = spins[
                    best_raw_replica
                ].detach().clone()

            should_log = (
                step == 0
                or (step + 1) % LOG_EVERY == 0
                or step == N_STEPS - 1
            )

            if should_log:
                mean_R = float(
                    R.mean().detach().cpu()
                )

                mean_abs_rx = float(
                    rx.abs().mean().detach().cpu()
                )

                mean_abs_rz = float(
                    rz.abs().mean().detach().cpu()
                )

                frac_polarized = float(
                    (rz.abs() > 0.95).to(DTYPE).mean().detach().cpu()
                )

                frac_ambiguous = float(
                    (rz.abs() < 0.05).to(DTYPE).mean().detach().cpu()
                )

                mean_expected_cut = float(
                    aux["expected_cut"].mean().detach().cpu()
                )

                max_expected_cut = float(
                    aux["expected_cut"].max().detach().cpu()
                )

                mean_entropy = float(
                    aux["entropy"].mean().detach().cpu()
                )

                mean_driver_energy = float(
                    aux["driver_energy"].mean().detach().cpu()
                )

                mean_free_energy = float(
                    aux["free_energy"].mean().detach().cpu()
                )

                log_rows.append(
                    {
                        "graph": graph,
                        "step": step + 1,
                        "temperature": float(
                            T_schedule[step].detach().cpu()
                        ),
                        "beta": float(
                            beta_t.detach().cpu()
                        ),
                        "gamma": float(
                            gamma_t.detach().cpu()
                        ),
                        "loss": float(
                            loss.detach().cpu()
                        ),
                        "mean_expected_cut": mean_expected_cut,
                        "max_expected_cut": max_expected_cut,
                        "step_best_raw_cut": step_best_cut_value,
                        "global_best_raw_cut_so_far": best_raw_cut_value,
                        "mean_entropy": mean_entropy,
                        "mean_driver_energy": mean_driver_energy,
                        "mean_free_energy": mean_free_energy,
                        "mean_R": mean_R,
                        "mean_abs_rx": mean_abs_rx,
                        "mean_abs_rz": mean_abs_rz,
                        "frac_polarized": frac_polarized,
                        "frac_ambiguous": frac_ambiguous,
                    }
                )

                current_best_accuracy = 100.0 * best_raw_cut_value / best_known_cut
                
                print(
                    f"{graph} | "
                    f"step {step + 1:5d}/{N_STEPS} | "
                    f"raw_best {best_raw_cut_value:.1f} | "
                    f"acc {current_best_accuracy:.4f}% | "
                    f"step_best {step_best_cut_value:.1f} | "
                    f"gamma {float(gamma_t.detach().cpu()):.5f} | "
                    f"T {float(T_schedule[step].detach().cpu()):.5e} | "
                    f"|rz| {mean_abs_rz:.4f}"
                )

    runtime_sec = time.time() - start_time

    assert best_raw_spins is not None, (
        "No raw best spin solution was recorded."
    )

    raw_best_cut = best_raw_cut_value
    raw_gap = best_known_cut - raw_best_cut
    raw_percent_accuracy = 100.0 * raw_best_cut / best_known_cut

    raw_delta_vs_v1 = raw_best_cut - v1_best_cut
    raw_gap_delta_vs_v1 = v1_gap - raw_gap
    raw_accuracy_delta_vs_v1 = raw_percent_accuracy - v1_percent_accuracy


    # --------------------------------------------------------
    # Optional polish
    # --------------------------------------------------------

    if USE_POLISH:
        candidate_spins = best_raw_spins.unsqueeze(0)

        polished_spins, polished_cut_tensor, polish_sweeps_done = polish_spins_for_graph(
            graph_data=graph_data,
            spins=candidate_spins,
            max_sweeps=POLISH_MAX_SWEEPS,
        )

        polished_best_cut = float(
            polished_cut_tensor.squeeze(0).detach().cpu()
        )

        polished_best_spins = polished_spins.squeeze(0).detach().clone()

    else:
        polished_best_cut = raw_best_cut
        polished_best_spins = best_raw_spins.detach().clone()
        polish_sweeps_done = 0

    polished_gap = best_known_cut - polished_best_cut
    polished_percent_accuracy = 100.0 * polished_best_cut / best_known_cut

    polished_delta_vs_v1 = polished_best_cut - v1_best_cut
    polished_gap_delta_vs_v1 = v1_gap - polished_gap
    polished_accuracy_delta_vs_v1 = polished_percent_accuracy - v1_percent_accuracy


    # --------------------------------------------------------
    # Result row
    # --------------------------------------------------------

    result_row = {
        "instance": graph,
        "nodes": N_g,
        "edges": graph_data["info"]["M"],
        "storage_mode": storage_mode,
        "n_steps": N_STEPS,
        "replicas": N_REPLICAS,
        "lr": ADAM_LR,

        "best_value": best_known_cut,

        "v1_best_cut": v1_best_cut,
        "v1_gap": v1_gap,
        "v1_percent_accuracy": v1_percent_accuracy,

        "raw_qefem_best_cut": raw_best_cut,
        "raw_qefem_gap": raw_gap,
        "raw_qefem_percent_accuracy": raw_percent_accuracy,
        "raw_qefem_best_step": best_raw_step,
        "raw_qefem_best_replica": best_raw_replica,

        "raw_delta_vs_v1": raw_delta_vs_v1,
        "raw_gap_delta_vs_v1": raw_gap_delta_vs_v1,
        "raw_accuracy_delta_vs_v1": raw_accuracy_delta_vs_v1,

        "use_polish": USE_POLISH,
        "polish_top_k": POLISH_TOP_K,
        "polish_max_sweeps": POLISH_MAX_SWEEPS,
        "polish_sweeps_done": polish_sweeps_done,

        "polished_best_cut": polished_best_cut,
        "polished_gap": polished_gap,
        "polished_percent_accuracy": polished_percent_accuracy,

        "polished_delta_vs_v1": polished_delta_vs_v1,
        "polished_gap_delta_vs_v1": polished_gap_delta_vs_v1,
        "polished_accuracy_delta_vs_v1": polished_accuracy_delta_vs_v1,

        "runtime_sec": runtime_sec,
        "seed": SEED + seed_offset,
    }

    log_df = pd.DataFrame(log_rows)

    print("\nFinished graph:", graph)
    print("runtime_sec:", runtime_sec)
    
    print("\nV1:")
    print("v1_best_cut:", v1_best_cut)
    print("v1_gap:", v1_gap)
    print("v1_percent_accuracy:", v1_percent_accuracy)
    
    print("\nRaw QEFEM:")
    print("raw_qefem_best_cut:", raw_best_cut)
    print("raw_qefem_gap:", raw_gap)
    print("raw_qefem_percent_accuracy:", raw_percent_accuracy)
    print("raw_delta_vs_v1:", raw_delta_vs_v1)
    print("raw_accuracy_delta_vs_v1:", raw_accuracy_delta_vs_v1)
    
    print("\nPolished QEFEM:")
    print("polished_best_cut:", polished_best_cut)
    print("polished_gap:", polished_gap)
    print("polished_percent_accuracy:", polished_percent_accuracy)
    print("polished_delta_vs_v1:", polished_delta_vs_v1)
    print("polished_accuracy_delta_vs_v1:", polished_accuracy_delta_vs_v1)

    return result_row, log_df, best_raw_spins, polished_best_spins


print("=" * 80)
print("SINGLE-GRAPH QEFEM RUNNER READY")
print("=" * 80)

print("Available:")
print("hard_cut_for_graph_data(graph_data, spins)")
print("qefem_free_energy_for_graph_data(graph_data, raw_R, theta, beta, gamma)")
print("run_qefem_on_single_graph(...)")

SINGLE-GRAPH QEFEM RUNNER READY
Available:
hard_cut_for_graph_data(graph_data, spins)
qefem_free_energy_for_graph_data(graph_data, raw_R, theta, beta, gamma)
run_qefem_on_single_graph(...)


In [50]:
# ============================================================
# Cell 13: Run selected group and save outputs
# ============================================================

assert "SELECTED_RUN_GROUP" in globals(), (
    "Run Cell 11 first: SELECTED_RUN_GROUP is missing."
)

assert "selected_benchmark_df" in globals(), (
    "Run Cell 11 first: selected_benchmark_df is missing."
)

assert "selected_v1_df" in globals(), (
    "Run Cell 11 first: selected_v1_df is missing."
)

assert "run_qefem_on_single_graph" in globals(), (
    "Run Cell 12 first: run_qefem_on_single_graph is missing."
)

assert "schedule_df" in globals(), (
    "Run Cell 08 first: schedule_df is missing."
)


# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

QEFEM_V2_OUTPUT_DIR = Path("qefem_v2_outputs")
QEFEM_V2_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SELECTED_GROUP_OUTPUT_DIR = QEFEM_V2_OUTPUT_DIR / SELECTED_RUN_GROUP
SELECTED_GROUP_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("=" * 80)
print("RUNNING SELECTED GROUP")
print("=" * 80)

print("Selected run group:", SELECTED_RUN_GROUP)
print("Output directory:")
print(SELECTED_GROUP_OUTPUT_DIR.resolve())


# ------------------------------------------------------------
# Save schedule used for this run group
# ------------------------------------------------------------

schedule_output_path = SELECTED_GROUP_OUTPUT_DIR / "schedule.csv"

schedule_df.to_csv(
    schedule_output_path,
    index=False,
)

print("\nSaved schedule:")
print(schedule_output_path.resolve())


# ------------------------------------------------------------
# Build selected run table
# ------------------------------------------------------------

selected_run_df = selected_benchmark_df[
    [
        "partition",
        "graph",
        "graph_type",
        "storage_mode",
        "N",
        "M",
        "best_known_cut",
    ]
].rename(
    columns={
        "graph": "instance",
        "N": "nodes",
        "M": "edges",
        "best_known_cut": "best_value",
    }
).merge(
    selected_v1_df[
        [
            "instance",
            "best_cut",
            "gap",
            "percent_accuracy",
        ]
    ].rename(
        columns={
            "best_cut": "v1_best_cut",
            "gap": "v1_gap",
            "percent_accuracy": "v1_percent_accuracy",
        }
    ),
    on="instance",
    how="left",
)

selected_run_df["graph_number"] = (
    selected_run_df["instance"]
    .str.replace("G", "", regex=False)
    .astype(int)
)

selected_run_df = selected_run_df.sort_values(
    "graph_number"
).reset_index(drop=True)

print("\nSelected run table:")
display(selected_run_df)


# ------------------------------------------------------------
# Checks before launching the expensive nonsense
# ------------------------------------------------------------

assert len(selected_run_df) > 0, (
    "selected_run_df is empty."
)

assert selected_run_df["instance"].is_unique, (
    "Duplicate graph instances in selected_run_df."
)

assert selected_run_df["best_value"].notna().all(), (
    "Missing best-known values."
)

assert selected_run_df["v1_best_cut"].notna().all(), (
    "Missing V1 best cuts."
)

assert selected_run_df["v1_gap"].notna().all(), (
    "Missing V1 gaps."
)

assert selected_run_df["v1_percent_accuracy"].notna().all(), (
    "Missing V1 percent accuracies."
)


# ------------------------------------------------------------
# Run graphs one by one
# ------------------------------------------------------------

group_result_rows = []
group_log_dfs = []

raw_spin_output_dir = SELECTED_GROUP_OUTPUT_DIR / "raw_best_spins"
polished_spin_output_dir = SELECTED_GROUP_OUTPUT_DIR / "polished_best_spins"

raw_spin_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

polished_spin_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)


for row_idx, row in selected_run_df.iterrows():
    graph = row["instance"]
    storage_mode = row["storage_mode"]

    print("\n" + "#" * 80)
    print(f"GROUP {SELECTED_RUN_GROUP}: {graph}")
    print("#" * 80)

    result_row, log_df, best_raw_spins, polished_best_spins = run_qefem_on_single_graph(
        graph=graph,
        storage_mode=storage_mode,
        best_known_cut=float(row["best_value"]),
        v1_best_cut=float(row["v1_best_cut"]),
        v1_gap=float(row["v1_gap"]),
        v1_percent_accuracy=float(row["v1_percent_accuracy"]),
        seed_offset=row_idx,
    )

    result_row["run_group"] = SELECTED_RUN_GROUP
    result_row["partition"] = int(row["partition"])
    result_row["graph_type"] = row["graph_type"]

    group_result_rows.append(result_row)

    log_df["run_group"] = SELECTED_RUN_GROUP
    log_df["partition"] = int(row["partition"])
    log_df["graph_type"] = row["graph_type"]

    group_log_dfs.append(log_df)


    # --------------------------------------------------------
    # Save per-graph log
    # --------------------------------------------------------

    graph_log_path = SELECTED_GROUP_OUTPUT_DIR / f"{graph}_training_log.csv"

    log_df.to_csv(
        graph_log_path,
        index=False,
    )

    print("\nSaved graph log:")
    print(graph_log_path.resolve())


    # --------------------------------------------------------
    # Save best spins
    # --------------------------------------------------------

    raw_spin_path = raw_spin_output_dir / f"{graph}_raw_best_spins.pt"
    polished_spin_path = polished_spin_output_dir / f"{graph}_polished_best_spins.pt"

    torch.save(
        best_raw_spins.detach().cpu(),
        raw_spin_path,
    )

    torch.save(
        polished_best_spins.detach().cpu(),
        polished_spin_path,
    )

    print("Saved raw best spins:")
    print(raw_spin_path.resolve())

    print("Saved polished best spins:")
    print(polished_spin_path.resolve())


    # --------------------------------------------------------
    # Save partial results after each graph
    # --------------------------------------------------------

    partial_results_df = pd.DataFrame(group_result_rows)

    partial_results_path = SELECTED_GROUP_OUTPUT_DIR / "partial_results.csv"

    partial_results_df.to_csv(
        partial_results_path,
        index=False,
    )

    print("Saved partial results:")
    print(partial_results_path.resolve())


# ------------------------------------------------------------
# Consolidate group outputs
# ------------------------------------------------------------

group_results_df = pd.DataFrame(group_result_rows)

group_logs_df = pd.concat(
    group_log_dfs,
    ignore_index=True,
)

group_results_df["graph_number"] = (
    group_results_df["instance"]
    .str.replace("G", "", regex=False)
    .astype(int)
)

group_results_df = group_results_df.sort_values(
    "graph_number"
).reset_index(drop=True)

group_results_df = group_results_df.drop(
    columns=["graph_number"]
)


# ------------------------------------------------------------
# Save final group outputs
# ------------------------------------------------------------

group_results_path = SELECTED_GROUP_OUTPUT_DIR / "group_results.csv"
group_logs_path = SELECTED_GROUP_OUTPUT_DIR / "group_training_logs.csv"

group_results_df.to_csv(
    group_results_path,
    index=False,
)

group_logs_df.to_csv(
    group_logs_path,
    index=False,
)

print("\n" + "=" * 80)
print("GROUP RUN COMPLETE")
print("=" * 80)

print("Saved group results:")
print(group_results_path.resolve())

print("\nSaved group training logs:")
print(group_logs_path.resolve())


# ------------------------------------------------------------
# Display result summary
# ------------------------------------------------------------

print("\nGroup results:")
display(group_results_df)

summary_columns = [
    "instance",
    "best_value",
    "v1_best_cut",
    "v1_gap",
    "v1_percent_accuracy",
    "raw_qefem_best_cut",
    "raw_qefem_gap",
    "raw_qefem_percent_accuracy",
    "raw_delta_vs_v1",
    "polished_best_cut",
    "polished_gap",
    "polished_percent_accuracy",
    "polished_delta_vs_v1",
    "runtime_sec",
]

print("\nCompact comparison:")
display(
    group_results_df[
        summary_columns
    ]
)


# ------------------------------------------------------------
# Group-level summary
# ------------------------------------------------------------

print("\nGroup-level summary:")

print("graphs:", len(group_results_df))

print("\nRaw QEFEM:")
print("mean raw delta vs v1:", float(group_results_df["raw_delta_vs_v1"].mean()))
print("min raw delta vs v1 :", float(group_results_df["raw_delta_vs_v1"].min()))
print("max raw delta vs v1 :", float(group_results_df["raw_delta_vs_v1"].max()))
print("raw beats v1 count  :", int((group_results_df["raw_delta_vs_v1"] > 0).sum()))
print("raw ties v1 count   :", int((group_results_df["raw_delta_vs_v1"] == 0).sum()))
print("raw loses v1 count  :", int((group_results_df["raw_delta_vs_v1"] < 0).sum()))

print("\nPolished QEFEM:")
print("mean polished delta vs v1:", float(group_results_df["polished_delta_vs_v1"].mean()))
print("min polished delta vs v1 :", float(group_results_df["polished_delta_vs_v1"].min()))
print("max polished delta vs v1 :", float(group_results_df["polished_delta_vs_v1"].max()))
print("polished beats v1 count  :", int((group_results_df["polished_delta_vs_v1"] > 0).sum()))
print("polished ties v1 count   :", int((group_results_df["polished_delta_vs_v1"] == 0).sum()))
print("polished loses v1 count  :", int((group_results_df["polished_delta_vs_v1"] < 0).sum()))

print("\nGap-zero:")
print("raw gap-zero count     :", int((group_results_df["raw_qefem_gap"] == 0).sum()))
print("polished gap-zero count:", int((group_results_df["polished_gap"] == 0).sum()))

print("\nRuntime:")
print("total runtime sec:", float(group_results_df["runtime_sec"].sum()))
print("mean runtime sec :", float(group_results_df["runtime_sec"].mean()))


# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------

assert len(group_results_df) == len(selected_run_df), (
    "Group results row count does not match selected run table."
)

assert group_results_df["instance"].is_unique, (
    "Duplicate graph instances in group results."
)

assert group_results_df["raw_qefem_best_cut"].notna().all(), (
    "Missing raw QEFEM best cuts."
)

assert group_results_df["polished_best_cut"].notna().all(), (
    "Missing polished best cuts."
)

assert (
    group_results_df["polished_best_cut"] >= group_results_df["raw_qefem_best_cut"]
).all(), (
    "Polish decreased at least one cut. That should not happen."
)

assert group_results_path.exists(), (
    "group_results.csv was not saved."
)

assert group_logs_path.exists(), (
    "group_training_logs.csv was not saved."
)


print("\nCell 13 complete: selected group has been run and saved.")

RUNNING SELECTED GROUP
Selected run group: small_1_3
Output directory:
/Users/ronit/Desktop/QeFEM/main/maxcut_results/qefem_v2_outputs/small_1_3

Saved schedule:
/Users/ronit/Desktop/QeFEM/main/maxcut_results/qefem_v2_outputs/small_1_3/schedule.csv

Selected run table:


,partition,instance,graph_type,storage_mode,nodes,edges,best_value,v1_best_cut,v1_gap,v1_percent_accuracy,graph_number
0,1,G1,random,dense,800,19176,11624.0,11611.0,13.0,99.888162,1
1,1,G2,random,dense,800,19176,11620.0,11610.0,10.0,99.913941,2
2,1,G3,random,dense,800,19176,11622.0,11610.0,12.0,99.896748,3
3,1,G4,random,dense,800,19176,11646.0,11637.0,9.0,99.922720,4
4,1,G5,random,dense,800,19176,11631.0,11622.0,9.0,99.922621,5
5,1,G6,random,dense,800,19176,2178.0,2173.0,5.0,99.770432,6
6,1,G7,random,dense,800,19176,2006.0,1992.0,14.0,99.302094,7
7,1,G8,random,dense,800,19176,2005.0,1991.0,14.0,99.301746,8
8,1,G9,random,dense,800,19176,2054.0,2041.0,13.0,99.367089,9
9,1,G10,random,dense,800,19176,2000.0,1992.0,8.0,99.600000,10



################################################################################
GROUP small_1_3: G1
################################################################################
RUNNING QEFEM ON SINGLE GRAPH
graph: G1
storage_mode: dense
N: 800
best_known_cut: 11624.0
v1_best_cut: 11611.0
G1 | step     1/2000 | raw_best 9749.0 | acc 83.8696% | step_best 9749.0 | gamma 1.00000 | T 1.16000e+00 | |rz| 0.0001
G1 | step    50/2000 | raw_best 9959.0 | acc 85.6762% | step_best 9959.0 | gamma 0.97549 | T 1.13157e+00 | |rz| 0.0000
G1 | step   100/2000 | raw_best 10061.0 | acc 86.5537% | step_best 10030.0 | gamma 0.95048 | T 1.10255e+00 | |rz| 0.0000
G1 | step   150/2000 | raw_best 10147.0 | acc 87.2935% | step_best 9084.0 | gamma 0.92546 | T 1.07354e+00 | |rz| 0.0000
G1 | step   200/2000 | raw_best 10147.0 | acc 87.2935% | step_best 6573.0 | gamma 0.90045 | T 1.04453e+00 | |rz| 0.0001
G1 | step   250/2000 | raw_best 11549.0 | acc 99.3548% | step_best 11549.0 | gamma 0.87544 | T 1.01552e+00

,instance,nodes,edges,storage_mode,n_steps,replicas,lr,best_value,v1_best_cut,v1_gap,...,polished_gap,polished_percent_accuracy,polished_delta_vs_v1,polished_gap_delta_vs_v1,polished_accuracy_delta_vs_v1,runtime_sec,seed,run_group,partition,graph_type
0,G1,800,19176,dense,2000,128,0.01,11624.0,11611.0,13.0,...,15.0,99.870957,-2.0,-2.0,-1.720578e-02,9.669467,7,small_1_3,1,random
1,G2,800,19176,dense,2000,128,0.01,11620.0,11610.0,10.0,...,25.0,99.784854,-15.0,-15.0,-1.290878e-01,9.719211,8,small_1_3,1,random
2,G3,800,19176,dense,2000,128,0.01,11622.0,11610.0,12.0,...,9.0,99.922561,3.0,3.0,2.581311e-02,9.873484,9,small_1_3,1,random
3,G4,800,19176,dense,2000,128,0.01,11646.0,11637.0,9.0,...,0.0,100.000000,9.0,9.0,7.727975e-02,9.243056,10,small_1_3,1,random
4,G5,800,19176,dense,2000,128,0.01,11631.0,11622.0,9.0,...,5.0,99.957011,4.0,4.0,3.439085e-02,9.612968,11,small_1_3,1,random
5,G6,800,19176,dense,2000,128,0.01,2178.0,2173.0,5.0,...,3.0,99.862259,2.0,2.0,9.182736e-02,10.144321,12,small_1_3,1,random
6,G7,800,19176,dense,2000,128,0.01,2006.0,1992.0,14.0,...,12.0,99.401795,2.0,2.0,9.970090e-02,9.427659,13,small_1_3,1,random
7,G8,800,19176,dense,2000,128,0.01,2005.0,1991.0,14.0,...,11.0,99.451372,3.0,3.0,1.496259e-01,8.788947,14,small_1_3,1,random
8,G9,800,19176,dense,2000,128,0.01,2054.0,2041.0,13.0,...,4.0,99.805258,9.0,9.0,4.381694e-01,9.662068,15,small_1_3,1,random
9,G10,800,19176,dense,2000,128,0.01,2000.0,1992.0,8.0,...,1.0,99.950000,7.0,7.0,3.500000e-01,9.976988,16,small_1_3,1,random



Compact comparison:


,instance,best_value,v1_best_cut,v1_gap,v1_percent_accuracy,raw_qefem_best_cut,raw_qefem_gap,raw_qefem_percent_accuracy,raw_delta_vs_v1,polished_best_cut,polished_gap,polished_percent_accuracy,polished_delta_vs_v1,runtime_sec
0,G1,11624.0,11611.0,13.0,99.888162,11609.0,15.0,99.870957,-2.0,11609.0,15.0,99.870957,-2.0,9.669467
1,G2,11620.0,11610.0,10.0,99.913941,11595.0,25.0,99.784854,-15.0,11595.0,25.0,99.784854,-15.0,9.719211
2,G3,11622.0,11610.0,12.0,99.896748,11613.0,9.0,99.922561,3.0,11613.0,9.0,99.922561,3.0,9.873484
3,G4,11646.0,11637.0,9.0,99.922720,11646.0,0.0,100.000000,9.0,11646.0,0.0,100.000000,9.0,9.243056
4,G5,11631.0,11622.0,9.0,99.922621,11626.0,5.0,99.957011,4.0,11626.0,5.0,99.957011,4.0,9.612968
5,G6,2178.0,2173.0,5.0,99.770432,2175.0,3.0,99.862259,2.0,2175.0,3.0,99.862259,2.0,10.144321
6,G7,2006.0,1992.0,14.0,99.302094,1994.0,12.0,99.401795,2.0,1994.0,12.0,99.401795,2.0,9.427659
7,G8,2005.0,1991.0,14.0,99.301746,1994.0,11.0,99.451372,3.0,1994.0,11.0,99.451372,3.0,8.788947
8,G9,2054.0,2041.0,13.0,99.367089,2050.0,4.0,99.805258,9.0,2050.0,4.0,99.805258,9.0,9.662068
9,G10,2000.0,1992.0,8.0,99.600000,1999.0,1.0,99.950000,7.0,1999.0,1.0,99.950000,7.0,9.976988



Group-level summary:
graphs: 21

Raw QEFEM:
mean raw delta vs v1: 0.9523809523809523
min raw delta vs v1 : -15.0
max raw delta vs v1 : 9.0
raw beats v1 count  : 10
raw ties v1 count   : 6
raw loses v1 count  : 5

Polished QEFEM:
mean polished delta vs v1: 0.9523809523809523
min polished delta vs v1 : -15.0
max polished delta vs v1 : 9.0
polished beats v1 count  : 10
polished ties v1 count   : 6
polished loses v1 count  : 5

Gap-zero:
raw gap-zero count     : 1
polished gap-zero count: 1

Runtime:
total runtime sec: 199.92718696594238
mean runtime sec : 9.520342236473446

Cell 13 complete: selected group has been run and saved.


In [51]:
# ============================================================
# Cell 14: Aggregate V2 win / loss / tie vs V1
# ============================================================

assert "QEFEM_V2_OUTPUT_DIR" in globals(), (
    "Run Cell 13 first: QEFEM_V2_OUTPUT_DIR is missing."
)

assert "v1_competitor_df" in globals(), (
    "Run Cell 05 first: v1_competitor_df is missing."
)

assert "benchmark_df" in globals(), (
    "Run Cell 04 first: benchmark_df is missing."
)


# ------------------------------------------------------------
# Locate all completed group results
# ------------------------------------------------------------

completed_group_result_paths = sorted(
    QEFEM_V2_OUTPUT_DIR.glob("*/group_results.csv")
)

print("=" * 80)
print("AGGREGATING V2 WIN / LOSS / TIE VS V1")
print("=" * 80)

print("Output root:")
print(QEFEM_V2_OUTPUT_DIR.resolve())

print("\nCompleted group result files:")
for path in completed_group_result_paths:
    print(path.resolve())

assert len(completed_group_result_paths) > 0, (
    "No completed group_results.csv files found."
)


# ------------------------------------------------------------
# Load completed V2 group results
# ------------------------------------------------------------

completed_group_dfs = []

for path in completed_group_result_paths:
    df = pd.read_csv(path)
    df["source_file"] = str(path)
    completed_group_dfs.append(df)

v2_completed_df = pd.concat(
    completed_group_dfs,
    ignore_index=True,
)

v2_completed_df["graph_number"] = (
    v2_completed_df["instance"]
    .str.replace("G", "", regex=False)
    .astype(int)
)

v2_completed_df = v2_completed_df.sort_values(
    "graph_number"
).reset_index(drop=True)


# ------------------------------------------------------------
# Check duplicates
# ------------------------------------------------------------

duplicate_instances = (
    v2_completed_df[
        v2_completed_df["instance"].duplicated(keep=False)
    ]["instance"]
    .unique()
    .tolist()
)

print("\nDuplicate completed instances:")
print(duplicate_instances)

assert len(duplicate_instances) == 0, (
    f"Duplicate V2 results found for: {duplicate_instances}"
)


# ------------------------------------------------------------
# Build clean comparison table
# ------------------------------------------------------------

v2_vs_v1_df = benchmark_df[
    [
        "partition",
        "graph",
        "graph_type",
        "storage_mode",
        "N",
        "M",
        "best_known_cut",
    ]
].rename(
    columns={
        "graph": "instance",
        "N": "nodes",
        "M": "edges",
        "best_known_cut": "best_value",
    }
).merge(
    v1_competitor_df[
        [
            "instance",
            "best_cut",
            "gap",
            "percent_accuracy",
            "runtime_sec",
        ]
    ].rename(
        columns={
            "best_cut": "v1_best_cut",
            "gap": "v1_gap",
            "percent_accuracy": "v1_percent_accuracy",
            "runtime_sec": "v1_runtime_sec",
        }
    ),
    on="instance",
    how="left",
).merge(
    v2_completed_df[
        [
            "instance",
            "run_group",
            "n_steps",
            "replicas",
            "lr",
            "raw_qefem_best_cut",
            "raw_qefem_gap",
            "raw_qefem_percent_accuracy",
            "raw_delta_vs_v1",
            "polished_best_cut",
            "polished_gap",
            "polished_percent_accuracy",
            "polished_delta_vs_v1",
            "runtime_sec",
            "use_polish",
            "polish_sweeps_done",
            "source_file",
        ]
    ].rename(
        columns={
            "runtime_sec": "v2_runtime_sec",
        }
    ),
    on="instance",
    how="inner",
)

v2_vs_v1_df["graph_number"] = (
    v2_vs_v1_df["instance"]
    .str.replace("G", "", regex=False)
    .astype(int)
)

v2_vs_v1_df = v2_vs_v1_df.sort_values(
    "graph_number"
).reset_index(drop=True)


# ------------------------------------------------------------
# Win / loss / tie labels
# ------------------------------------------------------------

def comparison_label(delta):
    if delta > 0:
        return "win"
    elif delta < 0:
        return "loss"
    else:
        return "tie"


v2_vs_v1_df["raw_vs_v1"] = v2_vs_v1_df["raw_delta_vs_v1"].apply(
    comparison_label
)

v2_vs_v1_df["polished_vs_v1"] = v2_vs_v1_df["polished_delta_vs_v1"].apply(
    comparison_label
)


# ------------------------------------------------------------
# Aggregate summary table
# ------------------------------------------------------------

raw_wins = int((v2_vs_v1_df["raw_vs_v1"] == "win").sum())
raw_ties = int((v2_vs_v1_df["raw_vs_v1"] == "tie").sum())
raw_losses = int((v2_vs_v1_df["raw_vs_v1"] == "loss").sum())

polished_wins = int((v2_vs_v1_df["polished_vs_v1"] == "win").sum())
polished_ties = int((v2_vs_v1_df["polished_vs_v1"] == "tie").sum())
polished_losses = int((v2_vs_v1_df["polished_vs_v1"] == "loss").sum())

num_completed = len(v2_vs_v1_df)

aggregate_vs_v1_df = pd.DataFrame(
    [
        {
            "method": "raw_qefem",
            "completed_graphs": num_completed,
            "wins_vs_v1": raw_wins,
            "ties_vs_v1": raw_ties,
            "losses_vs_v1": raw_losses,
            "win_rate_percent": 100.0 * raw_wins / num_completed,
            "non_loss_rate_percent": 100.0 * (raw_wins + raw_ties) / num_completed,
            "mean_delta_vs_v1": v2_vs_v1_df["raw_delta_vs_v1"].mean(),
            "min_delta_vs_v1": v2_vs_v1_df["raw_delta_vs_v1"].min(),
            "max_delta_vs_v1": v2_vs_v1_df["raw_delta_vs_v1"].max(),
            "mean_accuracy_percent": v2_vs_v1_df["raw_qefem_percent_accuracy"].mean(),
            "gap_zero_count": int((v2_vs_v1_df["raw_qefem_gap"] == 0).sum()),
        },
        {
            "method": "polished_qefem",
            "completed_graphs": num_completed,
            "wins_vs_v1": polished_wins,
            "ties_vs_v1": polished_ties,
            "losses_vs_v1": polished_losses,
            "win_rate_percent": 100.0 * polished_wins / num_completed,
            "non_loss_rate_percent": 100.0 * (polished_wins + polished_ties) / num_completed,
            "mean_delta_vs_v1": v2_vs_v1_df["polished_delta_vs_v1"].mean(),
            "min_delta_vs_v1": v2_vs_v1_df["polished_delta_vs_v1"].min(),
            "max_delta_vs_v1": v2_vs_v1_df["polished_delta_vs_v1"].max(),
            "mean_accuracy_percent": v2_vs_v1_df["polished_percent_accuracy"].mean(),
            "gap_zero_count": int((v2_vs_v1_df["polished_gap"] == 0).sum()),
        },
    ]
)


# ------------------------------------------------------------
# Aggregate by run group
# ------------------------------------------------------------

group_vs_v1_rows = []

for run_group, group_df in v2_vs_v1_df.groupby("run_group"):
    group_vs_v1_rows.append(
        {
            "run_group": run_group,
            "graphs": len(group_df),

            "raw_wins": int((group_df["raw_vs_v1"] == "win").sum()),
            "raw_ties": int((group_df["raw_vs_v1"] == "tie").sum()),
            "raw_losses": int((group_df["raw_vs_v1"] == "loss").sum()),
            "raw_mean_delta": group_df["raw_delta_vs_v1"].mean(),
            "raw_gap_zero_count": int((group_df["raw_qefem_gap"] == 0).sum()),

            "polished_wins": int((group_df["polished_vs_v1"] == "win").sum()),
            "polished_ties": int((group_df["polished_vs_v1"] == "tie").sum()),
            "polished_losses": int((group_df["polished_vs_v1"] == "loss").sum()),
            "polished_mean_delta": group_df["polished_delta_vs_v1"].mean(),
            "polished_gap_zero_count": int((group_df["polished_gap"] == 0).sum()),
        }
    )

group_vs_v1_df = pd.DataFrame(group_vs_v1_rows)


# ------------------------------------------------------------
# Aggregate by graph type and storage mode
# ------------------------------------------------------------

type_vs_v1_df = v2_vs_v1_df.groupby(
    ["graph_type", "storage_mode"],
    as_index=False,
).agg(
    graphs=("instance", "count"),

    raw_wins=("raw_vs_v1", lambda x: int((x == "win").sum())),
    raw_ties=("raw_vs_v1", lambda x: int((x == "tie").sum())),
    raw_losses=("raw_vs_v1", lambda x: int((x == "loss").sum())),
    raw_mean_delta=("raw_delta_vs_v1", "mean"),

    polished_wins=("polished_vs_v1", lambda x: int((x == "win").sum())),
    polished_ties=("polished_vs_v1", lambda x: int((x == "tie").sum())),
    polished_losses=("polished_vs_v1", lambda x: int((x == "loss").sum())),
    polished_mean_delta=("polished_delta_vs_v1", "mean"),
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\nOverall aggregate vs V1:")
display(aggregate_vs_v1_df)

print("\nAggregate by run group:")
display(group_vs_v1_df)

print("\nAggregate by graph type / storage mode:")
display(type_vs_v1_df)

print("\nPer-graph comparison:")
display(
    v2_vs_v1_df[
        [
            "partition",
            "instance",
            "run_group",
            "graph_type",
            "storage_mode",
            "nodes",
            "edges",
            "best_value",

            "v1_best_cut",
            "v1_gap",
            "v1_percent_accuracy",

            "raw_qefem_best_cut",
            "raw_qefem_gap",
            "raw_qefem_percent_accuracy",
            "raw_delta_vs_v1",
            "raw_vs_v1",

            "polished_best_cut",
            "polished_gap",
            "polished_percent_accuracy",
            "polished_delta_vs_v1",
            "polished_vs_v1",

            "v2_runtime_sec",
        ]
    ]
)


# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

V2_AGGREGATE_VS_V1_PATH = QEFEM_V2_OUTPUT_DIR / "qefem_v2_aggregate_vs_v1.csv"
V2_GROUP_AGGREGATE_VS_V1_PATH = QEFEM_V2_OUTPUT_DIR / "qefem_v2_group_aggregate_vs_v1.csv"
V2_TYPE_AGGREGATE_VS_V1_PATH = QEFEM_V2_OUTPUT_DIR / "qefem_v2_type_aggregate_vs_v1.csv"
V2_PER_GRAPH_VS_V1_PATH = QEFEM_V2_OUTPUT_DIR / "qefem_v2_per_graph_vs_v1.csv"

aggregate_vs_v1_df.to_csv(
    V2_AGGREGATE_VS_V1_PATH,
    index=False,
)

group_vs_v1_df.to_csv(
    V2_GROUP_AGGREGATE_VS_V1_PATH,
    index=False,
)

type_vs_v1_df.to_csv(
    V2_TYPE_AGGREGATE_VS_V1_PATH,
    index=False,
)

v2_vs_v1_df.drop(
    columns=["graph_number"],
    errors="ignore",
).to_csv(
    V2_PER_GRAPH_VS_V1_PATH,
    index=False,
)


print("\nSaved aggregate vs V1:")
print(V2_AGGREGATE_VS_V1_PATH.resolve())

print("\nSaved group aggregate vs V1:")
print(V2_GROUP_AGGREGATE_VS_V1_PATH.resolve())

print("\nSaved type aggregate vs V1:")
print(V2_TYPE_AGGREGATE_VS_V1_PATH.resolve())

print("\nSaved per-graph comparison vs V1:")
print(V2_PER_GRAPH_VS_V1_PATH.resolve())


# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------

assert len(v2_vs_v1_df) == len(v2_completed_df), (
    "Comparison table row count does not match completed V2 result count."
)

assert v2_vs_v1_df["instance"].is_unique, (
    "Duplicate instances in v2_vs_v1_df."
)

assert (
    raw_wins + raw_ties + raw_losses == num_completed
), (
    "Raw win/tie/loss counts do not sum to completed graph count."
)

assert (
    polished_wins + polished_ties + polished_losses == num_completed
), (
    "Polished win/tie/loss counts do not sum to completed graph count."
)

assert (
    v2_vs_v1_df["polished_best_cut"] >= v2_vs_v1_df["raw_qefem_best_cut"]
).all(), (
    "Polish decreased at least one graph result."
)

assert V2_AGGREGATE_VS_V1_PATH.exists()
assert V2_GROUP_AGGREGATE_VS_V1_PATH.exists()
assert V2_TYPE_AGGREGATE_VS_V1_PATH.exists()
assert V2_PER_GRAPH_VS_V1_PATH.exists()


print("\nCell 14 complete: aggregate win/loss/tie vs V1 is ready.")

AGGREGATING V2 WIN / LOSS / TIE VS V1
Output root:
/Users/ronit/Desktop/QeFEM/main/maxcut_results/qefem_v2_outputs

Completed group result files:
/Users/ronit/Desktop/QeFEM/main/maxcut_results/qefem_v2_outputs/small_1_3/group_results.csv

Duplicate completed instances:
[]

Overall aggregate vs V1:


,method,completed_graphs,wins_vs_v1,ties_vs_v1,losses_vs_v1,win_rate_percent,non_loss_rate_percent,mean_delta_vs_v1,min_delta_vs_v1,max_delta_vs_v1,mean_accuracy_percent,gap_zero_count
0,raw_qefem,21,10,6,5,47.619048,76.190476,0.952381,-15.0,9.0,99.252574,1
1,polished_qefem,21,10,6,5,47.619048,76.190476,0.952381,-15.0,9.0,99.252574,1



Aggregate by run group:


,run_group,graphs,raw_wins,raw_ties,raw_losses,raw_mean_delta,raw_gap_zero_count,polished_wins,polished_ties,polished_losses,polished_mean_delta,polished_gap_zero_count
0,small_1_3,21,10,6,5,0.952381,1,10,6,5,0.952381,1



Aggregate by graph type / storage mode:


,graph_type,storage_mode,graphs,raw_wins,raw_ties,raw_losses,raw_mean_delta,polished_wins,polished_ties,polished_losses,polished_mean_delta
0,plain,dense,8,1,4,3,-0.500000,1,4,3,-0.500000
1,random,dense,10,8,0,2,2.200000,8,0,2,2.200000
2,toroidal,dense,3,1,2,0,0.666667,1,2,0,0.666667



Per-graph comparison:


,partition,instance,run_group,graph_type,storage_mode,nodes,edges,best_value,v1_best_cut,v1_gap,...,raw_qefem_gap,raw_qefem_percent_accuracy,raw_delta_vs_v1,raw_vs_v1,polished_best_cut,polished_gap,polished_percent_accuracy,polished_delta_vs_v1,polished_vs_v1,v2_runtime_sec
0,1,G1,small_1_3,random,dense,800,19176,11624.0,11611.0,13.0,...,15.0,99.870957,-2.0,loss,11609.0,15.0,99.870957,-2.0,loss,9.669467
1,1,G2,small_1_3,random,dense,800,19176,11620.0,11610.0,10.0,...,25.0,99.784854,-15.0,loss,11595.0,25.0,99.784854,-15.0,loss,9.719211
2,1,G3,small_1_3,random,dense,800,19176,11622.0,11610.0,12.0,...,9.0,99.922561,3.0,win,11613.0,9.0,99.922561,3.0,win,9.873484
3,1,G4,small_1_3,random,dense,800,19176,11646.0,11637.0,9.0,...,0.0,100.000000,9.0,win,11646.0,0.0,100.000000,9.0,win,9.243056
4,1,G5,small_1_3,random,dense,800,19176,11631.0,11622.0,9.0,...,5.0,99.957011,4.0,win,11626.0,5.0,99.957011,4.0,win,9.612968
5,1,G6,small_1_3,random,dense,800,19176,2178.0,2173.0,5.0,...,3.0,99.862259,2.0,win,2175.0,3.0,99.862259,2.0,win,10.144321
6,1,G7,small_1_3,random,dense,800,19176,2006.0,1992.0,14.0,...,12.0,99.401795,2.0,win,1994.0,12.0,99.401795,2.0,win,9.427659
7,1,G8,small_1_3,random,dense,800,19176,2005.0,1991.0,14.0,...,11.0,99.451372,3.0,win,1994.0,11.0,99.451372,3.0,win,8.788947
8,1,G9,small_1_3,random,dense,800,19176,2054.0,2041.0,13.0,...,4.0,99.805258,9.0,win,2050.0,4.0,99.805258,9.0,win,9.662068
9,1,G10,small_1_3,random,dense,800,19176,2000.0,1992.0,8.0,...,1.0,99.950000,7.0,win,1999.0,1.0,99.950000,7.0,win,9.976988



Saved aggregate vs V1:
/Users/ronit/Desktop/QeFEM/main/maxcut_results/qefem_v2_outputs/qefem_v2_aggregate_vs_v1.csv

Saved group aggregate vs V1:
/Users/ronit/Desktop/QeFEM/main/maxcut_results/qefem_v2_outputs/qefem_v2_group_aggregate_vs_v1.csv

Saved type aggregate vs V1:
/Users/ronit/Desktop/QeFEM/main/maxcut_results/qefem_v2_outputs/qefem_v2_type_aggregate_vs_v1.csv

Saved per-graph comparison vs V1:
/Users/ronit/Desktop/QeFEM/main/maxcut_results/qefem_v2_outputs/qefem_v2_per_graph_vs_v1.csv

Cell 14 complete: aggregate win/loss/tie vs V1 is ready.


In [52]:
# ============================================================
# Cell 15: Print aggregate result summary
# ============================================================

assert "aggregate_vs_v1_df" in globals(), (
    "Run Cell 14 first: aggregate_vs_v1_df is missing."
)

assert "group_vs_v1_df" in globals(), (
    "Run Cell 14 first: group_vs_v1_df is missing."
)

assert "type_vs_v1_df" in globals(), (
    "Run Cell 14 first: type_vs_v1_df is missing."
)

assert "v2_vs_v1_df" in globals(), (
    "Run Cell 14 first: v2_vs_v1_df is missing."
)


print("=" * 80)
print("OVERALL V2 VS V1 AGGREGATE")
print("=" * 80)

display(aggregate_vs_v1_df)


print("\n" + "=" * 80)
print("GROUPWISE V2 VS V1 AGGREGATE")
print("=" * 80)

display(group_vs_v1_df)


print("\n" + "=" * 80)
print("GRAPH-TYPE / STORAGE-MODE AGGREGATE")
print("=" * 80)

display(type_vs_v1_df)


print("\n" + "=" * 80)
print("COMPACT PER-GRAPH RESULT")
print("=" * 80)

compact_result_df = v2_vs_v1_df[
    [
        "instance",
        "run_group",
        "graph_type",
        "storage_mode",
        "best_value",

        "v1_best_cut",
        "v1_gap",
        "v1_percent_accuracy",

        "raw_qefem_best_cut",
        "raw_qefem_gap",
        "raw_qefem_percent_accuracy",
        "raw_delta_vs_v1",
        "raw_vs_v1",

        "polished_best_cut",
        "polished_gap",
        "polished_percent_accuracy",
        "polished_delta_vs_v1",
        "polished_vs_v1",

        "v2_runtime_sec",
    ]
].copy()

display(compact_result_df)


print("\n" + "=" * 80)
print("ONE-LINE VERDICT")
print("=" * 80)

raw_row = aggregate_vs_v1_df[
    aggregate_vs_v1_df["method"] == "raw_qefem"
].iloc[0]

polished_row = aggregate_vs_v1_df[
    aggregate_vs_v1_df["method"] == "polished_qefem"
].iloc[0]

print(
    f"Raw QEFEM: "
    f"{int(raw_row['wins_vs_v1'])} wins, "
    f"{int(raw_row['ties_vs_v1'])} ties, "
    f"{int(raw_row['losses_vs_v1'])} losses "
    f"out of {int(raw_row['completed_graphs'])} completed graphs."
)

print(
    f"Polished QEFEM: "
    f"{int(polished_row['wins_vs_v1'])} wins, "
    f"{int(polished_row['ties_vs_v1'])} ties, "
    f"{int(polished_row['losses_vs_v1'])} losses "
    f"out of {int(polished_row['completed_graphs'])} completed graphs."
)

print(
    f"\nRaw mean delta vs V1: "
    f"{raw_row['mean_delta_vs_v1']:.4f}"
)

print(
    f"Polished mean delta vs V1: "
    f"{polished_row['mean_delta_vs_v1']:.4f}"
)

print(
    f"\nRaw mean accuracy: "
    f"{raw_row['mean_accuracy_percent']:.6f}%"
)

print(
    f"Polished mean accuracy: "
    f"{polished_row['mean_accuracy_percent']:.6f}%"
)

print(
    f"\nRaw gap-zero count: "
    f"{int(raw_row['gap_zero_count'])}"
)

print(
    f"Polished gap-zero count: "
    f"{int(polished_row['gap_zero_count'])}"
)


print("\nCell 15 complete: aggregate results printed.")

OVERALL V2 VS V1 AGGREGATE


,method,completed_graphs,wins_vs_v1,ties_vs_v1,losses_vs_v1,win_rate_percent,non_loss_rate_percent,mean_delta_vs_v1,min_delta_vs_v1,max_delta_vs_v1,mean_accuracy_percent,gap_zero_count
0,raw_qefem,21,10,6,5,47.619048,76.190476,0.952381,-15.0,9.0,99.252574,1
1,polished_qefem,21,10,6,5,47.619048,76.190476,0.952381,-15.0,9.0,99.252574,1



GROUPWISE V2 VS V1 AGGREGATE


,run_group,graphs,raw_wins,raw_ties,raw_losses,raw_mean_delta,raw_gap_zero_count,polished_wins,polished_ties,polished_losses,polished_mean_delta,polished_gap_zero_count
0,small_1_3,21,10,6,5,0.952381,1,10,6,5,0.952381,1



GRAPH-TYPE / STORAGE-MODE AGGREGATE


,graph_type,storage_mode,graphs,raw_wins,raw_ties,raw_losses,raw_mean_delta,polished_wins,polished_ties,polished_losses,polished_mean_delta
0,plain,dense,8,1,4,3,-0.500000,1,4,3,-0.500000
1,random,dense,10,8,0,2,2.200000,8,0,2,2.200000
2,toroidal,dense,3,1,2,0,0.666667,1,2,0,0.666667



COMPACT PER-GRAPH RESULT


,instance,run_group,graph_type,storage_mode,best_value,v1_best_cut,v1_gap,v1_percent_accuracy,raw_qefem_best_cut,raw_qefem_gap,raw_qefem_percent_accuracy,raw_delta_vs_v1,raw_vs_v1,polished_best_cut,polished_gap,polished_percent_accuracy,polished_delta_vs_v1,polished_vs_v1,v2_runtime_sec
0,G1,small_1_3,random,dense,11624.0,11611.0,13.0,99.888162,11609.0,15.0,99.870957,-2.0,loss,11609.0,15.0,99.870957,-2.0,loss,9.669467
1,G2,small_1_3,random,dense,11620.0,11610.0,10.0,99.913941,11595.0,25.0,99.784854,-15.0,loss,11595.0,25.0,99.784854,-15.0,loss,9.719211
2,G3,small_1_3,random,dense,11622.0,11610.0,12.0,99.896748,11613.0,9.0,99.922561,3.0,win,11613.0,9.0,99.922561,3.0,win,9.873484
3,G4,small_1_3,random,dense,11646.0,11637.0,9.0,99.922720,11646.0,0.0,100.000000,9.0,win,11646.0,0.0,100.000000,9.0,win,9.243056
4,G5,small_1_3,random,dense,11631.0,11622.0,9.0,99.922621,11626.0,5.0,99.957011,4.0,win,11626.0,5.0,99.957011,4.0,win,9.612968
5,G6,small_1_3,random,dense,2178.0,2173.0,5.0,99.770432,2175.0,3.0,99.862259,2.0,win,2175.0,3.0,99.862259,2.0,win,10.144321
6,G7,small_1_3,random,dense,2006.0,1992.0,14.0,99.302094,1994.0,12.0,99.401795,2.0,win,1994.0,12.0,99.401795,2.0,win,9.427659
7,G8,small_1_3,random,dense,2005.0,1991.0,14.0,99.301746,1994.0,11.0,99.451372,3.0,win,1994.0,11.0,99.451372,3.0,win,8.788947
8,G9,small_1_3,random,dense,2054.0,2041.0,13.0,99.367089,2050.0,4.0,99.805258,9.0,win,2050.0,4.0,99.805258,9.0,win,9.662068
9,G10,small_1_3,random,dense,2000.0,1992.0,8.0,99.600000,1999.0,1.0,99.950000,7.0,win,1999.0,1.0,99.950000,7.0,win,9.976988



ONE-LINE VERDICT
Raw QEFEM: 10 wins, 6 ties, 5 losses out of 21 completed graphs.
Polished QEFEM: 10 wins, 6 ties, 5 losses out of 21 completed graphs.

Raw mean delta vs V1: 0.9524
Polished mean delta vs V1: 0.9524

Raw mean accuracy: 99.252574%
Polished mean accuracy: 99.252574%

Raw gap-zero count: 1
Polished gap-zero count: 1

Cell 15 complete: aggregate results printed.


In [53]:
# ============================================================
# Cell 15: Print aggregate V2 vs V1 results
# ============================================================

assert "aggregate_vs_v1_df" in globals(), (
    "Run Cell 14 first: aggregate_vs_v1_df is missing."
)

assert "group_vs_v1_df" in globals(), (
    "Run Cell 14 first: group_vs_v1_df is missing."
)

assert "type_vs_v1_df" in globals(), (
    "Run Cell 14 first: type_vs_v1_df is missing."
)

assert "v2_vs_v1_df" in globals(), (
    "Run Cell 14 first: v2_vs_v1_df is missing."
)


# ------------------------------------------------------------
# Overall aggregate
# ------------------------------------------------------------

print("=" * 80)
print("OVERALL V2 VS V1 AGGREGATE")
print("=" * 80)

display(aggregate_vs_v1_df)


# ------------------------------------------------------------
# Extract raw and polished rows
# ------------------------------------------------------------

raw_row = aggregate_vs_v1_df[
    aggregate_vs_v1_df["method"] == "raw_qefem"
].iloc[0]

polished_row = aggregate_vs_v1_df[
    aggregate_vs_v1_df["method"] == "polished_qefem"
].iloc[0]


# ------------------------------------------------------------
# One-line verdict
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ONE-LINE VERDICT")
print("=" * 80)

print(
    f"Raw QEFEM vs V1: "
    f"{int(raw_row['wins_vs_v1'])} wins, "
    f"{int(raw_row['ties_vs_v1'])} ties, "
    f"{int(raw_row['losses_vs_v1'])} losses "
    f"out of {int(raw_row['completed_graphs'])} completed graphs."
)

print(
    f"Polished QEFEM vs V1: "
    f"{int(polished_row['wins_vs_v1'])} wins, "
    f"{int(polished_row['ties_vs_v1'])} ties, "
    f"{int(polished_row['losses_vs_v1'])} losses "
    f"out of {int(polished_row['completed_graphs'])} completed graphs."
)


# ------------------------------------------------------------
# Accuracy and delta summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ACCURACY / DELTA SUMMARY")
print("=" * 80)

print("\nRaw QEFEM:")
print("mean delta vs V1:", float(raw_row["mean_delta_vs_v1"]))
print("min delta vs V1 :", float(raw_row["min_delta_vs_v1"]))
print("max delta vs V1 :", float(raw_row["max_delta_vs_v1"]))
print("mean accuracy % :", float(raw_row["mean_accuracy_percent"]))
print("win rate %      :", float(raw_row["win_rate_percent"]))
print("non-loss rate % :", float(raw_row["non_loss_rate_percent"]))
print("gap-zero count  :", int(raw_row["gap_zero_count"]))

print("\nPolished QEFEM:")
print("mean delta vs V1:", float(polished_row["mean_delta_vs_v1"]))
print("min delta vs V1 :", float(polished_row["min_delta_vs_v1"]))
print("max delta vs V1 :", float(polished_row["max_delta_vs_v1"]))
print("mean accuracy % :", float(polished_row["mean_accuracy_percent"]))
print("win rate %      :", float(polished_row["win_rate_percent"]))
print("non-loss rate % :", float(polished_row["non_loss_rate_percent"]))
print("gap-zero count  :", int(polished_row["gap_zero_count"]))


# ------------------------------------------------------------
# Groupwise aggregate
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("GROUPWISE V2 VS V1 AGGREGATE")
print("=" * 80)

display(group_vs_v1_df)


# ------------------------------------------------------------
# Graph-type / storage-mode aggregate
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("GRAPH-TYPE / STORAGE-MODE AGGREGATE")
print("=" * 80)

display(type_vs_v1_df)


# ------------------------------------------------------------
# Compact per-graph comparison
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPACT PER-GRAPH COMPARISON")
print("=" * 80)

compact_v2_vs_v1_df = v2_vs_v1_df[
    [
        "partition",
        "instance",
        "run_group",
        "graph_type",
        "storage_mode",
        "best_value",

        "v1_best_cut",
        "v1_gap",
        "v1_percent_accuracy",

        "raw_qefem_best_cut",
        "raw_qefem_gap",
        "raw_qefem_percent_accuracy",
        "raw_delta_vs_v1",
        "raw_vs_v1",

        "polished_best_cut",
        "polished_gap",
        "polished_percent_accuracy",
        "polished_delta_vs_v1",
        "polished_vs_v1",

        "v2_runtime_sec",
    ]
].copy()

display(compact_v2_vs_v1_df)


# ------------------------------------------------------------
# Best and worst V2 movements
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BEST / WORST MOVEMENTS VS V1")
print("=" * 80)

raw_best_improvements_df = compact_v2_vs_v1_df.sort_values(
    "raw_delta_vs_v1",
    ascending=False,
).head(10)

raw_worst_losses_df = compact_v2_vs_v1_df.sort_values(
    "raw_delta_vs_v1",
    ascending=True,
).head(10)

polished_best_improvements_df = compact_v2_vs_v1_df.sort_values(
    "polished_delta_vs_v1",
    ascending=False,
).head(10)

polished_worst_losses_df = compact_v2_vs_v1_df.sort_values(
    "polished_delta_vs_v1",
    ascending=True,
).head(10)

print("\nTop raw improvements vs V1:")
display(
    raw_best_improvements_df[
        [
            "instance",
            "v1_best_cut",
            "raw_qefem_best_cut",
            "raw_delta_vs_v1",
            "raw_qefem_percent_accuracy",
            "raw_vs_v1",
        ]
    ]
)

print("\nWorst raw losses vs V1:")
display(
    raw_worst_losses_df[
        [
            "instance",
            "v1_best_cut",
            "raw_qefem_best_cut",
            "raw_delta_vs_v1",
            "raw_qefem_percent_accuracy",
            "raw_vs_v1",
        ]
    ]
)

print("\nTop polished improvements vs V1:")
display(
    polished_best_improvements_df[
        [
            "instance",
            "v1_best_cut",
            "polished_best_cut",
            "polished_delta_vs_v1",
            "polished_percent_accuracy",
            "polished_vs_v1",
        ]
    ]
)

print("\nWorst polished losses vs V1:")
display(
    polished_worst_losses_df[
        [
            "instance",
            "v1_best_cut",
            "polished_best_cut",
            "polished_delta_vs_v1",
            "polished_percent_accuracy",
            "polished_vs_v1",
        ]
    ]
)


# ------------------------------------------------------------
# Save compact summary
# ------------------------------------------------------------

COMPACT_V2_VS_V1_PATH = QEFEM_V2_OUTPUT_DIR / "qefem_v2_compact_vs_v1_summary.csv"

compact_v2_vs_v1_df.to_csv(
    COMPACT_V2_VS_V1_PATH,
    index=False,
)

print("\nSaved compact V2 vs V1 summary:")
print(COMPACT_V2_VS_V1_PATH.resolve())

assert COMPACT_V2_VS_V1_PATH.exists(), (
    "Compact V2 vs V1 summary was not saved."
)


print("\nCell 15 complete: aggregate V2 vs V1 results printed and saved.")

OVERALL V2 VS V1 AGGREGATE


,method,completed_graphs,wins_vs_v1,ties_vs_v1,losses_vs_v1,win_rate_percent,non_loss_rate_percent,mean_delta_vs_v1,min_delta_vs_v1,max_delta_vs_v1,mean_accuracy_percent,gap_zero_count
0,raw_qefem,21,10,6,5,47.619048,76.190476,0.952381,-15.0,9.0,99.252574,1
1,polished_qefem,21,10,6,5,47.619048,76.190476,0.952381,-15.0,9.0,99.252574,1



ONE-LINE VERDICT
Raw QEFEM vs V1: 10 wins, 6 ties, 5 losses out of 21 completed graphs.
Polished QEFEM vs V1: 10 wins, 6 ties, 5 losses out of 21 completed graphs.

ACCURACY / DELTA SUMMARY

Raw QEFEM:
mean delta vs V1: 0.9523809523809523
min delta vs V1 : -15.0
max delta vs V1 : 9.0
mean accuracy % : 99.25257381633546
win rate %      : 47.61904761904762
non-loss rate % : 76.19047619047619
gap-zero count  : 1

Polished QEFEM:
mean delta vs V1: 0.9523809523809523
min delta vs V1 : -15.0
max delta vs V1 : 9.0
mean accuracy % : 99.25257381633546
win rate %      : 47.61904761904762
non-loss rate % : 76.19047619047619
gap-zero count  : 1

GROUPWISE V2 VS V1 AGGREGATE


,run_group,graphs,raw_wins,raw_ties,raw_losses,raw_mean_delta,raw_gap_zero_count,polished_wins,polished_ties,polished_losses,polished_mean_delta,polished_gap_zero_count
0,small_1_3,21,10,6,5,0.952381,1,10,6,5,0.952381,1



GRAPH-TYPE / STORAGE-MODE AGGREGATE


...


Top polished improvements vs V1:


,instance,v1_best_cut,polished_best_cut,polished_delta_vs_v1,polished_percent_accuracy,polished_vs_v1
3,G4,11637.0,11646.0,9.0,100.000000,win
8,G9,2041.0,2050.0,9.0,99.805258,win
9,G10,1992.0,1999.0,7.0,99.950000,win
4,G5,11622.0,11626.0,4.0,99.957011,win
2,G3,11610.0,11613.0,3.0,99.922561,win
7,G8,1991.0,1994.0,3.0,99.451372,win
14,G15,3025.0,3027.0,2.0,99.245902,win
5,G6,2173.0,2175.0,2.0,99.862259,win
6,G7,1992.0,1994.0,2.0,99.401795,win
11,G12,550.0,552.0,2.0,99.280576,win



Worst polished losses vs V1:


,instance,v1_best_cut,polished_best_cut,polished_delta_vs_v1,polished_percent_accuracy,polished_vs_v1
1,G2,11610.0,11595.0,-15.0,99.784854,loss
17,G18,978.0,975.0,-3.0,98.286290,loss
0,G1,11611.0,11609.0,-2.0,99.870957,loss
15,G16,3027.0,3025.0,-2.0,99.115334,loss
13,G14,3040.0,3039.0,-1.0,99.184073,loss
18,G19,876.0,876.0,0.0,96.688742,tie
16,G17,3021.0,3021.0,0.0,99.146702,tie
12,G13,580.0,580.0,0.0,99.656357,tie
19,G20,930.0,930.0,0.0,98.831031,tie
10,G11,556.0,556.0,0.0,98.581560,tie



Saved compact V2 vs V1 summary:
/Users/ronit/Desktop/QeFEM/main/maxcut_results/qefem_v2_outputs/qefem_v2_compact_vs_v1_summary.csv

Cell 15 complete: aggregate V2 vs V1 results printed and saved.
